# Forecasting Competitive Football with NGBoost
## Pre-match classification, goal-margin regression, and in-play forecasting

This notebook is the modeling and evaluation stage of the course project. It consumes the leakage-audited **gold** tables produced by Phase 1 and implements all three predictive deliverables:

1. **Task C — pre-match outcome classification:** one row per match, outputting calibrated probabilities for Away / Draw / Home.
2. **Task R — pre-match goal-margin regression:** the same row, outputting home goals minus away goals, clipped to $[-5,5]$.
3. **Task L — in-play prediction:** one row per match snapshot, outputting updated outcome probabilities and final margin.

NGBoost is the paper method. The notebook also runs every library baseline required by the brief, performs calibration, imbalance experiments, uncertainty evaluation, market comparison, kernel-scaling analysis, per-minute live evaluation, ablation, SHAP explanations, and artifact export.

> **How to use this notebook:** in Google Colab choose **Runtime → Run all**. For local CPU execution, select the repository's `.venv` kernel or run `python code/modeling/run_notebook_local.py` from the activated environment. Read the Markdown immediately before each code cell. The default `standard` mode runs every required experiment with a modest validation budget; use `full` for the final report.

## 0. What is and is not repeated here

The upstream pipeline already created and audited:

- 380 Premier League 2015/16 matches;
- one pre-match row per match;
- 21 snapshots per match (7,980 rows total);
- a chronological 257 / 77 / 46 match train / validation / test split;
- de-vigged 1X2 market probabilities with 100% coverage;
- 12/12 leakage tests and 15/15 readiness checks.

The completed P1 stage then implemented Berrar, Lopes, and Dubitzky (2024):

- 15 historical EPL seasons were joined causally to the StatsBomb target season;
- Pearson recency selection was performed on training data only and frozen at $n=28$;
- the exact six-feature `total` and 18-feature `homeaway` contracts were generated;
- 374 matches were P1-eligible, split 251 / 77 / 46;
- all 15 P1 readiness checks, 13 P1 leakage checks, and 10 EXP01 checks passed;
- a fixed-model representation experiment selected **MD1 + P1 Home/Away** on validation RPS before test evaluation.

This notebook **does not rebuild or re-select Phase 1**. It verifies the saved readiness evidence, loads the frozen 146-feature pre-match representation, and carries the same 18 P1 values into every live snapshot of a match. The latter is an explicit project extension because the Berrar paper is pre-match only. The original 128-feature MD1 results remain visible as the controlled baseline, and the held-out reversal is reported rather than used to change the frozen choice.

## 1. Install or verify the reproducible environment

The exact versions below are intentionally pinned and include the paper authors' maintained `ngboost` package. In Colab this cell installs them. On a local CPU it performs a fast version check instead, so a previously prepared `.venv` is reused without downloading dependencies again. If a local package is missing, follow the command in the error message once and rerun the notebook.

The original multivariate paper used `ngboost==0.3.10`; this notebook uses the current compatible implementation while preserving the paper's distribution, log score, natural-gradient flag, and tree base learner. The fidelity table later makes this version difference explicit.

In [ ]:
import importlib.metadata as _environment_metadata
import subprocess as _environment_subprocess
import sys as _environment_sys

PINNED_ENVIRONMENT = {
    "numpy": "2.2.6",
    "pandas": "2.3.3",
    "scipy": "1.15.3",
    "matplotlib": "3.10.9",
    "joblib": "1.5.3",
    "pyarrow": "20.0.0",
    "scikit-learn": "1.7.2",
    "xgboost": "3.2.0",
    "lightgbm": "4.6.0",
    "ngboost": "0.5.11",
    "imbalanced-learn": "0.14.0",
    "shap": "0.48.0",
    "seaborn": "0.13.2",
    "psutil": "7.0.0",
    "Jinja2": "3.1.6",
}

try:
    import google.colab  # type: ignore  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    specifications = [f"{name}=={version}" for name, version in PINNED_ENVIRONMENT.items()]
    _environment_subprocess.check_call([
        _environment_sys.executable, "-m", "pip", "install", "-q", *specifications
    ])
    print("Colab dependencies installed. Restart the runtime once if Colab requests it.")
else:
    mismatches = []
    for distribution, required_version in PINNED_ENVIRONMENT.items():
        try:
            actual_version = _environment_metadata.version(distribution)
        except _environment_metadata.PackageNotFoundError:
            actual_version = "missing"
        if actual_version != required_version:
            mismatches.append(f"{distribution}: expected {required_version}, found {actual_version}")

    if mismatches:
        details = "\n  - ".join(mismatches)
        raise RuntimeError(
            "The local kernel does not match the pinned environment:\n  - " + details
            + "\nRun once from the repository root:\n"
            + r"  .\.venv\Scripts\python.exe -m pip install -r code\modeling\requirements_colab_ngboost.txt"
        )
    print(f"Local pinned environment verified with {_environment_sys.executable}")

## 2. Imports, switches, and fixed randomness

`RUN_MODE` controls only the number of hyperparameter candidates, never which required model families are evaluated.

- `quick`: one candidate per family; useful for checking that all cells work.
- `standard`: two candidates per family; the recommended first complete run.
- `full`: three candidates per family; use for final reported numbers.

The validation split selects configurations and fits the probability calibrators. The test split is touched only once, after those decisions. This is a chronological holdout design rather than shuffled cross-validation because future football matches must never help tune predictions for earlier matches.

In [ ]:
from __future__ import annotations

import copy
import gc
import importlib.metadata as ilm
import json
import math
import os
import platform
import random
import subprocess
import sys
import threading
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import seaborn as sns
import shap
import sklearn
import xgboost as xgb
from IPython.display import Markdown, display
from imblearn.over_sampling import ADASYN, BorderlineSMOTE, SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier, LGBMRegressor
from ngboost import NGBClassifier, NGBRegressor
from ngboost.distns import MultivariateNormal, Normal, k_categorical
from ngboost.scores import LogScore
from scipy.stats import chi2, norm, pearsonr
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.kernel_approximation import Nystroem
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

SEED = 42
RUN_MODE = "full"           # "quick", "standard", or "full"
RUN_SHAP = True             # Set False only for a fast plumbing check.
RUN_MULTIVARIATE = True     # Faithful O'Malley et al. paper experiment.
SAVE_MODELS = True
N_JOBS = -1
N_CANDIDATES = {"quick": 1, "standard": 2, "full": 3}[RUN_MODE]

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

print(f"Python {sys.version.split()[0]} | sklearn {sklearn.__version__} | mode={RUN_MODE}")

## 3. Locate the repository and load the frozen MD1 + P1 outputs

The path resolver supports the repository's documented Google Drive location, a Colab clone/upload, and local execution. If your Drive folder has a different name, set `PROJECT_ROOT_OVERRIDE` to it.

The modeling stage consumes both immutable data deliverables:

- MD1 `gold_prematch.parquet`, `gold_snapshots.parquet`, and `feature_dictionary.csv`;
- EXP01's validation-selected `prematch_md1_plus_p1_homeaway.parquet`;
- P1 feature contracts, readiness checks, frozen decision, and saved representation metrics.

The MD1 dictionary remains the allow-list for its 128 pre-match and 189 live features. The 18 P1 columns come from the separately audited P1 feature contract. Combining two positive allow-lists is safer than maintaining a hand-written block-list: new audit, odds, metadata, or label columns cannot silently enter the model.

In [ ]:
# Mounting is harmless outside Colab because the import simply fails.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

PROJECT_ROOT_OVERRIDE = ""  # Example: "/content/drive/MyDrive/ml_project"
RUN_FOLDER = "Premier_League_2015_16_DATA_ONLY_FULL_REAL"
REL_RESULT = Path("code/data_pipeline/results") / RUN_FOLDER
P1_EXPERIMENT_REL = Path("code/data_pipeline/results/EXP01_P1_REPRESENTATION_TASK_C")
P1_RUN_REL = Path(
    "code/data_pipeline/extracted/md1_data_pipeline_code/colab_results/"
    "EPL_2000_01_2015_16_Berrar_P1_HYBRID_REAL"
)

candidates = [
    Path(PROJECT_ROOT_OVERRIDE) if PROJECT_ROOT_OVERRIDE else None,
    Path("/content/drive/MyDrive/ml_project"),
    Path("/content/ML-Project-Football-Forecasting"),
    Path.cwd(),
    Path.cwd().parent,
]

PROJECT_ROOT = None
for candidate in [p for p in candidates if p is not None]:
    if (candidate / REL_RESULT / "data/gold/gold_prematch.parquet").exists():
        PROJECT_ROOT = candidate.resolve()
        break

if PROJECT_ROOT is None:
    searched = "\n".join(f"  - {p}" for p in candidates if p is not None)
    raise FileNotFoundError(
        "Could not locate the Phase 1 gold files. Set PROJECT_ROOT_OVERRIDE.\n"
        f"Searched:\n{searched}"
    )

RESULT_ROOT = PROJECT_ROOT / REL_RESULT
GOLD = RESULT_ROOT / "data/gold"
AUDIT = RESULT_ROOT / "audit"
P1_EXPERIMENT_ROOT = PROJECT_ROOT / P1_EXPERIMENT_REL
P1_DATA = P1_EXPERIMENT_ROOT / "data"
P1_AUDIT = P1_EXPERIMENT_ROOT / "audit"
P1_RUN_ROOT = PROJECT_ROOT / P1_RUN_REL
P1_RUN_AUDIT = P1_RUN_ROOT / "audit"
OUTPUT_DIR = PROJECT_ROOT / f"code/modeling/outputs/ngboost_p1_corrected_{RUN_MODE}"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
MODEL_DIR = OUTPUT_DIR / "models"
for folder in (OUTPUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

required_p1_files = [
    P1_DATA / "prematch_md1_plus_p1_homeaway.parquet",
    P1_DATA / "prematch_md1_common.parquet",
    P1_AUDIT / "feature_sets.json",
    P1_AUDIT / "representation_decision_validation.json",
    P1_AUDIT / "experiment_summary.json",
    P1_AUDIT / "exp01_readiness.csv",
    P1_RUN_AUDIT / "p1_readiness.csv",
    P1_RUN_AUDIT / "p1_leakage_tests.csv",
    P1_RUN_AUDIT / "p1_fidelity_table.csv",
    P1_EXPERIMENT_ROOT / "metrics/validation_metrics.csv",
    P1_EXPERIMENT_ROOT / "metrics/test_metrics.csv",
]
missing_p1_files = [str(path) for path in required_p1_files if not path.exists()]
if missing_p1_files:
    raise FileNotFoundError(
        "The frozen P1/EXP01 deliverable is incomplete:\n  - "
        + "\n  - ".join(missing_p1_files)
    )

prematch_md1_full = pd.read_parquet(GOLD / "gold_prematch.parquet").sort_values(
    "kickoff"
).reset_index(drop=True)
snapshots_md1_full = pd.read_parquet(GOLD / "gold_snapshots.parquet").sort_values(
    ["match_id", "snapshot_rank"]
).reset_index(drop=True)
feature_dictionary = pd.read_csv(GOLD / "feature_dictionary.csv")

with (P1_AUDIT / "feature_sets.json").open(encoding="utf-8") as handle:
    p1_feature_sets = json.load(handle)
with (P1_AUDIT / "representation_decision_validation.json").open(encoding="utf-8") as handle:
    p1_validation_decision = json.load(handle)
with (P1_AUDIT / "experiment_summary.json").open(encoding="utf-8") as handle:
    p1_experiment_summary = json.load(handle)

exp01_readiness = pd.read_csv(P1_AUDIT / "exp01_readiness.csv")
p1_readiness = pd.read_csv(P1_RUN_AUDIT / "p1_readiness.csv")
p1_leakage = pd.read_csv(P1_RUN_AUDIT / "p1_leakage_tests.csv")
p1_fidelity = pd.read_csv(P1_RUN_AUDIT / "p1_fidelity_table.csv")
p1_validation_metrics = pd.read_csv(P1_EXPERIMENT_ROOT / "metrics/validation_metrics.csv")
p1_test_metrics = pd.read_csv(P1_EXPERIMENT_ROOT / "metrics/test_metrics.csv")
p1_augmented_saved = pd.read_parquet(P1_DATA / "prematch_md1_plus_p1_homeaway.parquet")
p1_md1_common_saved = pd.read_parquet(P1_DATA / "prematch_md1_common.parquet")

p1_features = list(p1_feature_sets["P1_HOMEAWAY_SELECTED"])
p1_payload = p1_augmented_saved[["match_id", *p1_features]].copy()

# Preserve all MD1 task metadata, then restrict to the frozen P1-eligible IDs.
prematch = prematch_md1_full.merge(
    p1_payload, on="match_id", how="inner", validate="one_to_one"
).sort_values("kickoff").reset_index(drop=True)
snapshots = snapshots_md1_full.merge(
    p1_payload, on="match_id", how="inner", validate="many_to_one"
).sort_values(["match_id", "snapshot_rank"]).reset_index(drop=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MD1 full:", prematch_md1_full.shape, snapshots_md1_full.shape)
print("P1-eligible modeling rows:", prematch.shape, snapshots.shape)
print("Frozen representation:", p1_validation_decision["preferred_project_representation_by_validation_rps"])
display(prematch.head(3))
display(snapshots.head(3))

### 3.1 Read the upstream P1 evidence before modeling

This is a reporting and integrity step, not another model-selection step. EXP01 used one fixed Random Forest for every representation so that only the feature view changed. It froze `MD1_PLUS_P1_HOMEAWAY` using validation RPS. Test results later favored the 128-feature MD1 baseline, but the paired uncertainty intervals did not establish a robust P1 improvement. We preserve both facts:

1. use the validation-frozen 146-feature representation for the downstream model suite;
2. report the held-out reversal honestly and never use it to re-select the representation.

The market is shown for context only and remains excluded from every feature matrix.

In [ ]:
def audit_boolean(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "pass"})

assert p1_experiment_summary["status"] == "PASS"
assert p1_experiment_summary["selected_n"] == 28
assert p1_experiment_summary["common_matches"] == 374
assert p1_validation_decision["selection_basis"].startswith("validation RPS only")
assert p1_validation_decision["preferred_project_representation_by_validation_rps"] == "MD1_PLUS_P1_HOMEAWAY"
assert len(p1_features) == 18
assert set(p1_features).issubset(p1_augmented_saved.columns)
assert audit_boolean(p1_readiness["passed"]).all()
assert audit_boolean(p1_leakage["passed"]).all()
assert audit_boolean(exp01_readiness["passed"]).all()

p1_representation_summary = pd.concat([
    p1_validation_metrics.assign(stage="validation"),
    p1_test_metrics.assign(stage="test (report only)"),
], ignore_index=True)

display(Markdown("### Frozen P1 decision"))
display(pd.Series(p1_validation_decision).to_frame("value"))
display(Markdown("### Upstream representation results"))
display(p1_representation_summary.sort_values(["stage", "rps"]).style.format({
    "rps": "{:.4f}", "log_loss": "{:.4f}", "brier": "{:.4f}",
    "accuracy": "{:.3f}", "ece": "{:.4f}",
}))
p1_representation_summary.to_csv(TABLE_DIR / "p1_upstream_representation_metrics.csv", index=False)
p1_readiness.to_csv(TABLE_DIR / "p1_upstream_readiness.csv", index=False)
p1_leakage.to_csv(TABLE_DIR / "p1_upstream_leakage_checks.csv", index=False)
p1_fidelity.to_csv(TABLE_DIR / "p1_upstream_fidelity_table.csv", index=False)
exp01_readiness.to_csv(TABLE_DIR / "p1_exp01_readiness.csv", index=False)
display(Markdown("### P1 paper-fidelity decisions"))
display(p1_fidelity)
print(f"P1 readiness: {len(p1_readiness)}/{len(p1_readiness)} PASS")
print(f"P1 leakage checks: {len(p1_leakage)}/{len(p1_leakage)} PASS")
print(f"EXP01 readiness: {len(exp01_readiness)}/{len(exp01_readiness)} PASS")

## 4. Fail-fast data, split, and leakage contract

These assertions are not substitutes for the upstream audits; they protect this modeling run from a wrong folder, a partially regenerated table, or accidental split contamination.

The most important live-data assertion is match-level disjointness. A match's 21 highly correlated snapshots must never be divided across train and test. The audit columns also prove that the largest event order used by a snapshot does not cross its cutoff.

In [ ]:
EXPECTED_MD1_SPLITS = {"train": 257, "validation": 77, "test": 46}
EXPECTED_SPLITS = {"train": 251, "validation": 77, "test": 46}

assert prematch_md1_full["match_id"].is_unique
assert len(prematch_md1_full) == 380
assert prematch_md1_full.groupby("split").size().to_dict() == EXPECTED_MD1_SPLITS
assert snapshots_md1_full.groupby("match_id").size().eq(21).all()

assert prematch["match_id"].is_unique, "Expected exactly one row per P1-eligible match"
assert len(prematch) == 374, f"Expected 374 P1-eligible matches, found {len(prematch)}"
assert len(snapshots) == 374 * 21, f"Expected 7,854 P1-eligible snapshots, found {len(snapshots)}"
assert snapshots.duplicated(["match_id", "snapshot_id"]).sum() == 0
assert snapshots.groupby("match_id").size().eq(21).all(), "Every match must have 21 snapshots"
assert set(prematch["match_id"]) == set(snapshots["match_id"])
assert prematch.groupby("split").size().to_dict() == EXPECTED_SPLITS
assert snapshots.groupby("match_id")["split"].nunique().eq(1).all()
assert (snapshots["max_used_event_order"] <= snapshots["snapshot_cutoff_event_order"]).all()
assert set(prematch["label_outcome"].unique()) == {"A", "D", "H"}
assert prematch["label_margin"].between(-5, 5).all()
assert prematch.loc[prematch["split"].eq("test"), "odds_tagged"].eq(1).all()
assert np.allclose(
    prematch[["market_p_away", "market_p_draw", "market_p_home"]].sum(axis=1),
    1.0,
    atol=1e-7,
)

eligible = feature_dictionary["model_eligible"].astype(str).str.lower().eq("true")
md1_pre_features = feature_dictionary.loc[
    eligible & feature_dictionary["table"].eq("gold_prematch"), "column"
].tolist()
md1_live_features = feature_dictionary.loc[
    eligible & feature_dictionary["table"].eq("gold_snapshots"), "column"
].tolist()
pre_features = [*md1_pre_features, *p1_features]
live_features = [*md1_live_features, *p1_features]

assert set(pre_features).issubset(prematch.columns)
assert set(live_features).issubset(snapshots.columns)
assert len(md1_pre_features) == 128 and len(pre_features) == 146
assert len(md1_live_features) == 189 and len(live_features) == 207
assert len(pre_features) == len(set(pre_features))
assert len(live_features) == len(set(live_features))
assert set(prematch["match_id"]) == set(p1_md1_common_saved["match_id"])
assert snapshots.groupby("match_id")[p1_features].nunique(dropna=False).le(1).all().all()
for forbidden in ["label", "score", "odds", "market", "join"]:
    suspicious = [c for c in pre_features if forbidden in c.lower()]
    assert not suspicious, f"Forbidden pre-match feature pattern {forbidden}: {suspicious}"
assert not ({"final_home_score", "final_away_score", "home_score", "away_score"} & set(pre_features))

split_check = pd.DataFrame({
    "prematch_rows": prematch.groupby("split").size(),
    "snapshot_rows": snapshots.groupby("split").size(),
    "snapshot_matches": snapshots.groupby("split")["match_id"].nunique(),
})
display(split_check)
print(f"MD1 allow-list: {len(md1_pre_features)} pre-match, {len(md1_live_features)} live")
print(f"P1 extension: {len(p1_features)} frozen pre-match features")
print(f"Final matrices: {len(pre_features)} pre-match, {len(live_features)} live")
print("All fail-fast contracts passed.")

## 5. Formal task definitions and class order

We use the fixed ordinal class order

$$\text{Away win} < \text{Draw} < \text{Home win},$$

encoded as `A=0, D=1, H=2`. The order matters for Ranked Probability Score (RPS). It is never inferred alphabetically at evaluation time.

For a $K=3$ class forecast with cumulative predicted probability $F_k$ and cumulative one-hot observation $O_k$,

$$\operatorname{RPS}=\frac{1}{K-1}\sum_{k=1}^{K-1}(F_k-O_k)^2.$$

For regression, $y=\operatorname{clip}(G_H-G_A,-5,5)$. For a live row at time $t$, the label is the same match's final outcome/margin, but the predictors contain only the pre-match vector and the event prefix available by $t$.

In [ ]:
CLASS_NAMES = ["Away", "Draw", "Home"]
CLASS_CODES = ["A", "D", "H"]
CLASS_TO_INT = {"A": 0, "D": 1, "H": 2}
INT_TO_CLASS = {v: k for k, v in CLASS_TO_INT.items()}

def encode_outcome(series: pd.Series) -> np.ndarray:
    encoded = series.map(CLASS_TO_INT)
    assert encoded.notna().all()
    return encoded.to_numpy(dtype=int)

def one_hot(y: np.ndarray, n_classes: int = 3) -> np.ndarray:
    return np.eye(n_classes, dtype=float)[np.asarray(y, dtype=int)]

def normalize_proba(proba: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    p = np.clip(np.asarray(proba, dtype=float), eps, 1.0)
    return p / p.sum(axis=1, keepdims=True)

def ranked_probability_score(y_true: np.ndarray, proba: np.ndarray) -> float:
    p = normalize_proba(proba)
    observed = one_hot(y_true, p.shape[1])
    return float(np.mean(np.sum((np.cumsum(p, axis=1)[:, :-1] -
                                 np.cumsum(observed, axis=1)[:, :-1]) ** 2, axis=1)
                         / (p.shape[1] - 1)))

def multiclass_brier(y_true: np.ndarray, proba: np.ndarray) -> float:
    p = normalize_proba(proba)
    return float(np.mean(np.sum((p - one_hot(y_true, p.shape[1])) ** 2, axis=1)))

def expected_calibration_error(y_true: np.ndarray, proba: np.ndarray, n_bins: int = 10) -> float:
    # Top-label ECE: confidence versus correctness in equal-width bins.
    p = normalize_proba(proba)
    confidence = p.max(axis=1)
    prediction = p.argmax(axis=1)
    correct = prediction.eq(y_true) if isinstance(prediction, pd.Series) else prediction == y_true
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (confidence > lo) & (confidence <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())
    return float(ece)

def classification_metrics(y_true: np.ndarray, proba: np.ndarray) -> dict:
    p = normalize_proba(proba)
    pred = p.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, labels=[0, 1, 2], zero_division=0
    )
    return {
        "RPS": ranked_probability_score(y_true, p),
        "LogLoss": log_loss(y_true, p, labels=[0, 1, 2]),
        "Brier": multiclass_brier(y_true, p),
        "ECE": expected_calibration_error(y_true, p),
        "Accuracy": accuracy_score(y_true, pred),
        "MacroPrecision": precision.mean(),
        "MacroRecall": recall.mean(),
        "MacroF1": f1.mean(),
        **{f"F1_{name}": value for name, value in zip(CLASS_NAMES, f1)},
    }

def regression_metrics(y_true: np.ndarray, pred: np.ndarray) -> dict:
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(pred, dtype=float)
    correlation = np.nan if np.std(p) == 0 or np.std(y) == 0 else pearsonr(y, p).statistic
    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": mean_squared_error(y, p) ** 0.5,
        "Correlation": correlation,
    }

def reliability_points(y_true: np.ndarray, proba: np.ndarray, class_id: int, n_bins: int = 10):
    p = normalize_proba(proba)[:, class_id]
    observed = (np.asarray(y_true) == class_id).astype(float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (p > lo) & (p <= hi)
        if mask.any():
            rows.append({"mean_probability": p[mask].mean(), "observed_frequency": observed[mask].mean(),
                         "count": mask.sum(), "lower": lo, "upper": hi})
    return pd.DataFrame(rows)

def plot_reliability(y_true, before, after, title, save_name=None):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True, sharey=True)
    for class_id, (ax, class_name) in enumerate(zip(axes, CLASS_NAMES)):
        for probs, label, marker in [(before, "raw", "o"), (after, "Platt-calibrated", "s")]:
            points = reliability_points(y_true, probs, class_id)
            ax.plot(points["mean_probability"], points["observed_frequency"], marker=marker, label=label)
        ax.plot([0, 1], [0, 1], "k--", lw=1)
        ax.set_title(class_name)
        ax.set_xlabel("Mean predicted probability")
        ax.set_ylabel("Observed frequency")
        ax.legend()
    fig.suptitle(title)
    fig.tight_layout()
    if save_name:
        fig.savefig(FIG_DIR / save_name, dpi=180, bbox_inches="tight")
    plt.show()

print("Metric helpers ready. A perfect forecast has RPS/LogLoss/Brier/ECE = 0.")

## 6. NGBoost and the supplied paper: method and adaptation

O'Malley, Sykulski, Lumpkin, and Schuler (2023) extend NGBoost from a scalar response to a vector response $Y\in\mathbb{R}^P$. NGBoost predicts distribution parameters rather than only a point estimate. At boosting iteration $b$, weak regression trees approximate the **natural gradient** of a proper score:

$$\widetilde{\nabla}_{\theta}\,\ell(\theta)
= I(\theta)^{-1}\nabla_{\theta}\ell(\theta),$$

where $I(\theta)$ is the Fisher information. The Fisher preconditioner accounts for the geometry and scale of distribution parameters, making joint updates to location and uncertainty much more stable than ordinary gradients.

For the paper-faithful two-output football experiment, define

$$Y=(G_H,G_A)^\top\sim\mathcal N_2(\mu(X),\Sigma(X)).$$

The model therefore learns expected home goals, expected away goals, both variances, and their conditional covariance. Positive definiteness is enforced through a Cholesky factor of the precision matrix. We evaluate joint negative log-likelihood and interval coverage, and discretize fixed-seed samples to obtain Home/Draw/Away probabilities. This is an **adaptation** because football goals are non-negative counts rather than truly Gaussian.

For the mandatory direct tasks we additionally use:

- categorical NGBoost with `k_categorical(3)` and log score for Task C;
- univariate Normal NGBoost with log score for the clipped margin in Tasks R and L.

This gives direct task comparability while the separate multivariate cell reproduces the supplied paper's central contribution.

### 6.1 Hand derivation of the core update (report Appendix A material)

At the current prediction $\theta_i^{(b-1)}$, minimize negative log-likelihood $S_i=-\log p(y_i\mid\theta_i)$. Ordinary multiparameter boosting would fit trees to $-\nabla_\theta S_i$. But an ordinary gradient changes if the same distribution is reparameterized. The steepest direction measured in distribution space instead solves

$$\min_{\delta}\quad \nabla S_i^\top\delta
\quad\text{subject to}\quad
\delta^\top I(\theta_i)\delta=\varepsilon,$$

where the quadratic constraint is the local second-order approximation to KL divergence. A Lagrange multiplier gives

$$\delta\propto-I(\theta_i)^{-1}\nabla S_i
=-\widetilde{\nabla}_{\theta}S_i.$$

For iteration $b$:

1. compute $g_i^{(b)}=I(\theta_i^{(b-1)})^{-1}\nabla_\theta S_i$;
2. fit one regression tree per distribution parameter so $f^{(b)}(x_i)\approx g_i^{(b)}$;
3. choose a scalar $\rho^{(b)}$ by line search;
4. update

$$\theta_i^{(b)}=\theta_i^{(b-1)}-\eta\rho^{(b)}f^{(b)}(x_i).$$

For a univariate Normal parameterized by $(\mu,s=\log\sigma)$,

$$S=\tfrac12\log(2\pi)+s+\frac{(y-\mu)^2}{2e^{2s}},\qquad
\nabla S=\begin{bmatrix}(\mu-y)/\sigma^2\\1-(y-\mu)^2/\sigma^2\end{bmatrix},$$

and $I(\mu,s)=\operatorname{diag}(1/\sigma^2,2)$. Thus

$$\widetilde{\nabla}S
=\begin{bmatrix}\mu-y\\\tfrac12\{1-(y-\mu)^2/\sigma^2\}\end{bmatrix}.$$

The mean component becomes a residual in data units, while the scale component reacts to standardized squared residuals. This compact result explains why natural-gradient updates learn location and uncertainty together without the location coordinate dominating solely because of parameter scaling.

### 6.2 Fidelity table

| Paper element | Notebook implementation | Status / reason |
|---|---|---|
| Conditional multivariate Gaussian | `MultivariateNormal(2)` for `(home_goals, away_goals)` | Faithful distribution family |
| Heterogeneous covariance $\Sigma(X)$ | Estimated by multivariate NGBoost | Faithful central contribution |
| Log score / maximum likelihood | `LogScore` | Faithful |
| Natural gradients | `natural_gradient=True` | Faithful |
| Regression-tree base learners | depth-controlled `DecisionTreeRegressor` | Faithful family; tuned depth differs by dataset |
| Joint NLL and coverage | Evaluated on held-out football matches | Faithful metrics, new domain |
| Energy score | Monte Carlo estimator included below | Faithful multivariate proper score |
| Ocean velocities | Home and away goals | Necessary domain adaptation |
| Continuous Gaussian outputs | Rounded and clipped only when deriving 1X2 probabilities | Explicit approximation; raw joint metrics remain continuous |
| Paper package version 0.3.10 | Maintained pinned version 0.5.11 | Reproducibility/security update; API behavior recorded |

## 7. Freeze the chronological matrices

Every transformer lives inside a pipeline and is fitted only when the estimator sees the training split. The validation and test frames remain raw DataFrames until `predict`/`transform` time.

`X_pre` contains the validation-frozen 128 MD1 + 18 P1 representation. `X_live` contains the 189 audited live fields + the same 18 frozen P1 values. The `X_*_md1` matrices are retained for transparent feature-count checks and possible sensitivity work; they are not silently mixed with the primary arm.

No bookmaker column is in any feature matrix. Market probabilities remain an external reference, so the comparison does not let a model imitate its baseline.

In [ ]:
pre_split = {name: prematch[prematch["split"].eq(name)].copy() for name in EXPECTED_SPLITS}
live_split = {name: snapshots[snapshots["split"].eq(name)].copy() for name in EXPECTED_SPLITS}

X_pre = {name: frame[pre_features] for name, frame in pre_split.items()}
X_pre_md1 = {name: frame[md1_pre_features] for name, frame in pre_split.items()}
y_pre_c = {name: encode_outcome(frame["label_outcome"]) for name, frame in pre_split.items()}
y_pre_r = {name: frame["label_margin"].to_numpy(float) for name, frame in pre_split.items()}

X_live = {name: frame[live_features] for name, frame in live_split.items()}
X_live_md1 = {name: frame[md1_live_features] for name, frame in live_split.items()}
y_live_c = {name: encode_outcome(frame["label_outcome"]) for name, frame in live_split.items()}
y_live_r = {name: frame["label_margin"].to_numpy(float) for name, frame in live_split.items()}

market_test = pre_split["test"][["market_p_away", "market_p_draw", "market_p_home"]].to_numpy(float)
market_validation = pre_split["validation"][["market_p_away", "market_p_draw", "market_p_home"]].to_numpy(float)

print("Pre-match class counts")
display(pd.crosstab(prematch["split"], prematch["label_outcome"], margins=True))
print("Matrix shapes")
display(pd.DataFrame({
    "pre-match MD1+P1": {k: v.shape for k, v in X_pre.items()},
    "pre-match MD1 only": {k: v.shape for k, v in X_pre_md1.items()},
    "live MD1+P1": {k: v.shape for k, v in X_live.items()},
    "live MD1 only": {k: v.shape for k, v in X_live_md1.items()},
}))

## 8. Shared calibration, timing, memory, and model-selection utilities

`MulticlassPlattCalibrator` fits a multinomial logistic mapping from raw log-probabilities to calibrated probabilities using validation matches only. This is vector/Platt scaling: it preserves the three-way simplex and never observes test labels.

`fit_with_resources` samples resident memory in a background thread while fitting. The reported peak is process RSS and therefore includes the Python baseline; `PeakDeltaMB` is the additional observed peak. Native libraries allocate outside Python, which is why RSS is more useful here than `tracemalloc`.

Each family receives the same number of hand-specified candidates (`N_CANDIDATES`). Classification uses validation RPS, ordinary regression uses validation RMSE, and distributional NGBoost regression uses validation NLL. All configurations and scale diagnostics are retained in the tuning table—no test-set tuning and no hidden cherry-picking.

In [ ]:
class MulticlassPlattCalibrator:
    def __init__(self, seed=SEED):
        self.model = LogisticRegression(C=1.0, max_iter=5000, random_state=seed)

    def fit(self, raw_proba, y):
        self.model.fit(np.log(normalize_proba(raw_proba)), y)
        return self

    def predict_proba(self, raw_proba):
        return normalize_proba(self.model.predict_proba(np.log(normalize_proba(raw_proba))))

@dataclass
class FitResources:
    seconds: float
    peak_rss_mb: float
    peak_delta_mb: float

def fit_with_resources(estimator, X, y):
    process = psutil.Process(os.getpid())
    baseline = process.memory_info().rss
    peak = [baseline]
    stop = threading.Event()

    def monitor():
        while not stop.wait(0.02):
            peak[0] = max(peak[0], process.memory_info().rss)

    worker = threading.Thread(target=monitor, daemon=True)
    worker.start()
    started = time.perf_counter()
    try:
        fitted = clone(estimator).fit(X, y)
    finally:
        elapsed = time.perf_counter() - started
        peak[0] = max(peak[0], process.memory_info().rss)
        stop.set()
        worker.join(timeout=2)
    return fitted, FitResources(elapsed, peak[0] / 2**20, (peak[0] - baseline) / 2**20)

def align_proba(model, X):
    p = np.asarray(model.predict_proba(X), dtype=float)
    # NGBoost follows the encoded 0..K-1 order but does not expose classes_.
    classes = np.asarray(getattr(model, "classes_", np.arange(p.shape[1])), dtype=int)
    aligned = np.zeros((len(X), 3), dtype=float)
    aligned[:, classes] = p
    return normalize_proba(aligned)

def pipeline_pred_dist(model, X):
    transformed = model.named_steps["prep"].transform(X)
    return model.named_steps["model"].pred_dist(transformed)

def select_candidate(model_name, candidates, X_train, y_train, X_val, y_val, task):
    rows, fitted_models = [], []
    for candidate_id, estimator in enumerate(candidates[:N_CANDIDATES], start=1):
        fitted, resources = fit_with_resources(estimator, X_train, y_train)
        if task == "classification":
            pred = align_proba(fitted, X_val)
            score = ranked_probability_score(y_val, pred)
            criterion = "validation_RPS"
            validation_stats = {criterion: score}
        else:
            pred = np.asarray(fitted.predict(X_val), dtype=float)
            validation_rmse = mean_squared_error(y_val, pred) ** 0.5
            validation_stats = {"validation_RMSE": validation_rmse}
            if model_name == "NGBoost":
                # NGBoost is a distributional model. Selecting it solely by the
                # conditional mean can reward a catastrophically collapsed scale.
                # A proper log score evaluates location and uncertainty together.
                val_dist = pipeline_pred_dist(fitted, X_val)
                val_sigma = np.asarray(val_dist.params["scale"], dtype=float)
                val_nll = float(-np.mean(val_dist.logpdf(y_val)))
                z80 = norm.ppf(0.90)
                val_coverage80 = float(np.mean(
                    (y_val >= pred - z80 * val_sigma) & (y_val <= pred + z80 * val_sigma)
                ))
                validation_stats.update({
                    "validation_NLL": val_nll,
                    "validation_MinSigma": float(np.min(val_sigma)),
                    "validation_MedianSigma": float(np.median(val_sigma)),
                    "validation_Coverage80Raw": val_coverage80,
                })
                score = val_nll if np.isfinite(val_nll) else np.inf
                criterion = "validation_NLL"
            else:
                score = validation_rmse
                criterion = "validation_RMSE"
        rows.append({
            "Model": model_name, "Candidate": candidate_id,
            "SelectionCriterion": criterion, **validation_stats,
            "TrainSeconds": resources.seconds, "PeakRSSMB": resources.peak_rss_mb,
            "PeakDeltaMB": resources.peak_delta_mb,
            "Parameters": repr(estimator.get_params(deep=True)),
        })
        fitted_models.append(fitted)
        gc.collect()
    best_idx = int(np.argmin([r[criterion] for r in rows]))
    rows[best_idx]["Selected"] = True
    for i, row in enumerate(rows):
        row.setdefault("Selected", False)
    return fitted_models[best_idx], pd.DataFrame(rows), rows[best_idx]

compute_rows = []
tuning_tables = []
print("Shared experimental utilities ready.")

## 9. Candidate model factories

The brief requires Dummy, SVM, Random Forest, textbook GBM, XGBoost, LightGBM, and the paper method for classification; regression adds exact and approximate Kernel Ridge. Candidate lists change depth, regularization, or learning rate while keeping the search budget comparable.

Tree models need median imputation but not scaling. Kernel methods receive imputation and standardization fitted on training data. NGBoost uses shallow regression trees as its vector-valued base learner and explicitly enables the natural gradient.

In [ ]:
def tree_pipe(model):
    return Pipeline([
        ("prep", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", model),
    ])

def kernel_pipe(model):
    return Pipeline([
        ("prep", Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ])),
        ("model", model),
    ])

def classification_candidates(live=False):
    n_tree = [250, 400, 600]
    tree_depth = [2, 3, 4] if not live else [3, 4, 5]
    specs = {
        "Dummy": [tree_pipe(DummyClassifier(strategy="prior"))],
    }
    if not live:
        specs["SVC-RBF"] = [
            kernel_pipe(SVC(C=c, gamma="scale", probability=True, random_state=SEED))
            for c in [0.5, 1.0, 2.0]
        ]
    specs.update({
        "Random Forest": [
            tree_pipe(RandomForestClassifier(
                n_estimators=n_tree[i], max_depth=[None, 8, 12][i], min_samples_leaf=[4, 3, 2][i],
                max_features="sqrt", random_state=SEED, n_jobs=N_JOBS,
            )) for i in range(3)
        ],
        "GBM": [
            tree_pipe(GradientBoostingClassifier(
                n_estimators=n_tree[i], learning_rate=[0.03, 0.05, 0.08][i],
                max_depth=tree_depth[i], min_samples_leaf=[8, 6, 4][i], random_state=SEED,
            )) for i in range(3)
        ],
        "XGBoost": [
            tree_pipe(xgb.XGBClassifier(
                n_estimators=n_tree[i], learning_rate=[0.03, 0.05, 0.08][i],
                max_depth=tree_depth[i], min_child_weight=[8, 5, 3][i],
                subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                objective="multi:softprob", eval_metric="mlogloss", random_state=SEED,
                n_jobs=N_JOBS, tree_method="hist",
            )) for i in range(3)
        ],
        "LightGBM": [
            tree_pipe(LGBMClassifier(
                n_estimators=n_tree[i], learning_rate=[0.03, 0.05, 0.08][i],
                max_depth=tree_depth[i], num_leaves=[7, 15, 31][i],
                min_child_samples=[20, 15, 10][i], subsample=0.8, colsample_bytree=0.8,
                reg_lambda=2.0, random_state=SEED, n_jobs=N_JOBS, verbosity=-1,
            )) for i in range(3)
        ],
        "NGBoost": [
            tree_pipe(NGBClassifier(
                Dist=k_categorical(3), Score=LogScore,
                Base=DecisionTreeRegressor(
                    max_depth=tree_depth[i], min_samples_leaf=[12, 8, 5][i], random_state=SEED
                ),
                n_estimators=n_tree[i], learning_rate=[0.02, 0.03, 0.05][i],
                minibatch_frac=0.8, col_sample=0.8, natural_gradient=True,
                random_state=SEED, verbose=False,
            )) for i in range(3)
        ],
    })
    return specs

def regression_candidates(live=False):
    n_tree = [250, 400, 600]
    tree_depth = [2, 3, 4] if not live else [3, 4, 5]
    specs = {
        "Dummy": [tree_pipe(DummyRegressor(strategy="mean"))],
    }
    if not live:
        specs.update({
            "SVR-RBF": [kernel_pipe(SVR(C=c, epsilon=e, gamma="scale"))
                        for c, e in [(0.5, 0.2), (1.0, 0.15), (2.0, 0.1)]],
            "Kernel Ridge exact": [kernel_pipe(KernelRidge(alpha=a, kernel="rbf", gamma=g))
                                   for a, g in [(1.0, 0.01), (0.5, 0.01), (1.0, 0.03)]],
            "Kernel Ridge Nystroem": [Pipeline([
                ("prep", Pipeline([
                    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                    ("scaler", StandardScaler()),
                ])),
                ("nystroem", Nystroem(kernel="rbf", gamma=g, n_components=components, random_state=SEED)),
                ("model", Ridge(alpha=a)),
            ]) for a, g, components in [(1.0, 0.01, 80), (0.5, 0.01, 120), (1.0, 0.03, 160)]],
        })
    specs.update({
        "Random Forest": [
            tree_pipe(RandomForestRegressor(
                n_estimators=n_tree[i], max_depth=[None, 8, 12][i], min_samples_leaf=[4, 3, 2][i],
                max_features="sqrt", random_state=SEED, n_jobs=N_JOBS,
            )) for i in range(3)
        ],
        "GBM": [
            tree_pipe(GradientBoostingRegressor(
                n_estimators=n_tree[i], learning_rate=[0.03, 0.05, 0.08][i],
                max_depth=tree_depth[i], min_samples_leaf=[8, 6, 4][i], loss="huber",
                random_state=SEED,
            )) for i in range(3)
        ],
        "XGBoost": [
            tree_pipe(xgb.XGBRegressor(
                n_estimators=n_tree[i], learning_rate=[0.03, 0.05, 0.08][i],
                max_depth=tree_depth[i], min_child_weight=[8, 5, 3][i], subsample=0.8,
                colsample_bytree=0.8, reg_lambda=2.0, objective="reg:squarederror",
                eval_metric="rmse", random_state=SEED, n_jobs=N_JOBS, tree_method="hist",
            )) for i in range(3)
        ],
        "LightGBM": [
            tree_pipe(LGBMRegressor(
                n_estimators=n_tree[i], learning_rate=[0.03, 0.05, 0.08][i],
                max_depth=tree_depth[i], num_leaves=[7, 15, 31][i],
                min_child_samples=[20, 15, 10][i], subsample=0.8, colsample_bytree=0.8,
                reg_lambda=2.0, random_state=SEED, n_jobs=N_JOBS, verbosity=-1,
            )) for i in range(3)
        ],
        "NGBoost": [
            tree_pipe(NGBRegressor(
                Dist=Normal, Score=LogScore,
                Base=DecisionTreeRegressor(
                    max_depth=tree_depth[i], min_samples_leaf=[12, 8, 5][i], random_state=SEED
                ),
                n_estimators=n_tree[i], learning_rate=[0.02, 0.03, 0.05][i],
                minibatch_frac=0.8, col_sample=0.8, natural_gradient=True,
                random_state=SEED, verbose=False,
            )) for i in range(3)
        ],
    })
    return specs

display(pd.DataFrame({
    "Task C candidates": {k: min(len(v), N_CANDIDATES) for k, v in classification_candidates().items()},
    "Task R candidates": {k: min(len(v), N_CANDIDATES) for k, v in regression_candidates().items()},
}).fillna("—"))

# Part I — Task C: pre-match outcome probabilities

Each classifier is fitted on the 251 P1-eligible training matches using the validation-frozen 146-feature representation. Validation RPS selects its configuration, and validation predictions fit its Platt calibrator. The final table reports raw and calibrated test metrics so calibration cannot hide a loss in discrimination.

The market row uses the already de-vigged Football-Data probabilities on the **identical 46 test matches**. It is a reference, never a feature.

In [ ]:
cls_models = {}
cls_calibrators = {}
cls_val_raw = {}
cls_val_cal = {}
cls_test_raw = {}
cls_test_cal = {}

for model_name, candidates in classification_candidates(live=False).items():
    print(f"Fitting Task C: {model_name}")
    fitted, tuning, selected = select_candidate(
        model_name, candidates,
        X_pre["train"], y_pre_c["train"], X_pre["validation"], y_pre_c["validation"],
        task="classification",
    )
    tuning.insert(0, "Task", "C")
    tuning_tables.append(tuning)
    compute_rows.append({"Task": "C", **selected})
    cls_models[model_name] = fitted
    cls_val_raw[model_name] = align_proba(fitted, X_pre["validation"])
    cls_test_raw[model_name] = align_proba(fitted, X_pre["test"])
    calibrator = MulticlassPlattCalibrator().fit(cls_val_raw[model_name], y_pre_c["validation"])
    cls_calibrators[model_name] = calibrator
    cls_val_cal[model_name] = calibrator.predict_proba(cls_val_raw[model_name])
    cls_test_cal[model_name] = calibrator.predict_proba(cls_test_raw[model_name])

cls_rows = []
for model_name in cls_models:
    cls_rows.append({"Model": model_name, "Calibration": "Raw",
                     **classification_metrics(y_pre_c["test"], cls_test_raw[model_name])})
    cls_rows.append({"Model": model_name, "Calibration": "Platt",
                     **classification_metrics(y_pre_c["test"], cls_test_cal[model_name])})

cls_rows.append({"Model": "De-vigged market", "Calibration": "Market baseline",
                 **classification_metrics(y_pre_c["test"], market_test)})
task_c_results = pd.DataFrame(cls_rows).sort_values(["RPS", "LogLoss"]).reset_index(drop=True)
display(task_c_results.style.format(precision=4).highlight_min(
    subset=["RPS", "LogLoss", "Brier", "ECE"], color="#d8f3dc"
).highlight_max(subset=["Accuracy", "MacroF1"], color="#d8f3dc"))

task_c_results.to_csv(TABLE_DIR / "task_c_all_metrics.csv", index=False)
pd.concat(tuning_tables, ignore_index=True).to_csv(TABLE_DIR / "tuning_all_candidates_so_far.csv", index=False)

## 10. Calibration audit and reliability diagrams

For every probabilistic classifier, the solid/marker curve should approach the diagonal after calibration. ECE can improve while RPS or log-loss worsens; the table above is the authority, not the visual appearance alone. The test set is small, so wide or jagged reliability curves are expected and should be stated as uncertainty rather than smoothed away.

In [ ]:
calibration_delta = []
for model_name in cls_models:
    before = classification_metrics(y_pre_c["test"], cls_test_raw[model_name])
    after = classification_metrics(y_pre_c["test"], cls_test_cal[model_name])
    calibration_delta.append({
        "Model": model_name,
        "ECE_raw": before["ECE"], "ECE_Platt": after["ECE"],
        "Delta_ECE": after["ECE"] - before["ECE"],
        "RPS_raw": before["RPS"], "RPS_Platt": after["RPS"],
        "Delta_RPS": after["RPS"] - before["RPS"],
    })
    plot_reliability(
        y_pre_c["test"], cls_test_raw[model_name], cls_test_cal[model_name],
        f"Task C reliability — {model_name}",
        f"task_c_reliability_{model_name.lower().replace(' ', '_')}.png",
    )

calibration_delta = pd.DataFrame(calibration_delta).sort_values("ECE_Platt")
display(calibration_delta.style.format(precision=4))
calibration_delta.to_csv(TABLE_DIR / "task_c_calibration_delta.csv", index=False)

## 11. Outcome-level errors, confusion matrix, and ten worst pre-match forecasts

A low aggregate RPS can still conceal systematic draw failure. The per-class report exposes precision/recall/F1. The ten-worst table ranks NGBoost matches by the negative log-probability assigned to the true outcome; this prioritizes confidently wrong forecasts, exactly the cases that require a post-mortem and local SHAP explanation.

In [ ]:
paper_cls_name = "NGBoost"
paper_cls_probs = cls_test_cal[paper_cls_name]
paper_cls_pred = paper_cls_probs.argmax(axis=1)

report_c = pd.DataFrame(classification_report(
    y_pre_c["test"], paper_cls_pred, labels=[0, 1, 2], target_names=CLASS_NAMES,
    output_dict=True, zero_division=0,
)).T
display(report_c.style.format(precision=3))

cm = confusion_matrix(y_pre_c["test"], paper_cls_pred, labels=[0, 1, 2])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title("Task C — calibrated NGBoost confusion matrix")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(FIG_DIR / "task_c_ngboost_confusion.png", dpi=180, bbox_inches="tight")
plt.show()

true_probability = paper_cls_probs[np.arange(len(paper_cls_probs)), y_pre_c["test"]]
worst_c = pre_split["test"][[
    "match_id", "kickoff", "home_team_name", "away_team_name", "label_outcome"
]].copy()
worst_c[["p_away", "p_draw", "p_home"]] = paper_cls_probs
worst_c["predicted"] = [INT_TO_CLASS[i] for i in paper_cls_pred]
worst_c["case_log_loss"] = -np.log(np.clip(true_probability, 1e-12, 1.0))
worst_c = worst_c.nlargest(10, "case_log_loss")
display(worst_c)
worst_c.to_csv(TABLE_DIR / "task_c_ten_worst_ngboost.csv", index=False)

coverage_rate = pre_split["test"]["odds_tagged"].mean()
print(f"Market tagging coverage on the exact Task C test set: {coverage_rate:.1%} ({len(market_test)}/{len(market_test)})")

## 12. Required imbalance and resampling study

The same Random Forest configuration is used in five arms: vanilla, SMOTE, BorderlineSMOTE, ADASYN, and balanced class weights. Each sampler appears **inside an `imblearn` pipeline after training-only imputation**. The validation and test sets are never resampled.

This experiment is intentionally limited to pre-match classification. Interpolating snapshots from different matches would create physically meaningless pseudo-games and is explicitly forbidden by the brief.

In [ ]:
def imbalance_estimator(kind):
    prep = SimpleImputer(strategy="median", add_indicator=True)
    rf_kwargs = dict(
        n_estimators=400, max_depth=8, min_samples_leaf=3, max_features="sqrt",
        random_state=SEED, n_jobs=N_JOBS,
    )
    if kind == "Class weights":
        return ImbPipeline([("prep", prep), ("model", RandomForestClassifier(
            class_weight="balanced", **rf_kwargs
        ))])
    sampler = {
        "Vanilla": "passthrough",
        "SMOTE": SMOTE(random_state=SEED, k_neighbors=5),
        "BorderlineSMOTE": BorderlineSMOTE(random_state=SEED, k_neighbors=5),
        # With only mild three-class imbalance, ADASYN's default can round the
        # per-class allocation to zero. Target the true minority (Draw=1) explicitly.
        "ADASYN": ADASYN(random_state=SEED, n_neighbors=5, sampling_strategy={1: 100}),
    }[kind]
    return ImbPipeline([("prep", prep), ("sampler", sampler),
                        ("model", RandomForestClassifier(**rf_kwargs))])

imbalance_models = {}
imbalance_rows = []
imbalance_class_rows = []
for arm in ["Vanilla", "SMOTE", "BorderlineSMOTE", "ADASYN", "Class weights"]:
    print("Fitting imbalance arm:", arm)
    model, resources = fit_with_resources(imbalance_estimator(arm), X_pre["train"], y_pre_c["train"])
    val_raw = align_proba(model, X_pre["validation"])
    test_raw = align_proba(model, X_pre["test"])
    calibrator = MulticlassPlattCalibrator().fit(val_raw, y_pre_c["validation"])
    test_cal = calibrator.predict_proba(test_raw)
    imbalance_models[arm] = (model, calibrator)
    imbalance_rows.append({"Method": arm, **classification_metrics(y_pre_c["test"], test_cal),
                           "TrainSeconds": resources.seconds, "PeakRSSMB": resources.peak_rss_mb})
    p, r, f, support = precision_recall_fscore_support(
        y_pre_c["test"], test_cal.argmax(axis=1), labels=[0, 1, 2], zero_division=0
    )
    for class_name, precision, recall, f1, n in zip(CLASS_NAMES, p, r, f, support):
        imbalance_class_rows.append({"Method": arm, "Class": class_name,
                                     "Precision": precision, "Recall": recall, "F1": f1, "Support": n})
    plot_reliability(y_pre_c["test"], test_raw, test_cal, f"Imbalance arm — {arm}",
                     f"imbalance_reliability_{arm.lower().replace(' ', '_')}.png")

imbalance_results = pd.DataFrame(imbalance_rows).sort_values("RPS")
imbalance_per_class = pd.DataFrame(imbalance_class_rows)
display(imbalance_results.style.format(precision=4))
display(imbalance_per_class.pivot(index="Method", columns="Class", values=["Precision", "Recall", "F1"])
        .style.format(precision=3))
imbalance_results.to_csv(TABLE_DIR / "imbalance_aggregate.csv", index=False)
imbalance_per_class.to_csv(TABLE_DIR / "imbalance_per_class.csv", index=False)

# Part II — Task R: pre-match goal-margin regression

The regression target is the signed home-minus-away final margin clipped to $[-5,5]$. Validation RMSE selects ordinary point regressors. **NGBoost is selected by validation NLL**, a proper score that evaluates its predicted mean and scale together. This prevents a candidate with a marginally better mean but collapsed uncertainty from winning.

The required point table still reports MAE, RMSE, and Pearson correlation for every family. NGBoost additionally reports validation minimum/median scale and raw 80% coverage in the tuning table, followed by full held-out uncertainty diagnostics below.

In [ ]:
reg_models = {}
reg_val_pred = {}
reg_test_pred = {}

for model_name, candidates in regression_candidates(live=False).items():
    print(f"Fitting Task R: {model_name}")
    fitted, tuning, selected = select_candidate(
        model_name, candidates,
        X_pre["train"], y_pre_r["train"], X_pre["validation"], y_pre_r["validation"],
        task="regression",
    )
    tuning.insert(0, "Task", "R")
    tuning_tables.append(tuning)
    compute_rows.append({"Task": "R", **selected})
    reg_models[model_name] = fitted
    reg_val_pred[model_name] = np.asarray(fitted.predict(X_pre["validation"]), dtype=float)
    reg_test_pred[model_name] = np.asarray(fitted.predict(X_pre["test"]), dtype=float)

reg_rows = [{"Model": name, **regression_metrics(y_pre_r["test"], pred)}
            for name, pred in reg_test_pred.items()]
task_r_results = pd.DataFrame(reg_rows).sort_values("RMSE").reset_index(drop=True)
display(task_r_results.style.format(precision=4).highlight_min(
    subset=["MAE", "RMSE"], color="#d8f3dc"
).highlight_max(subset=["Correlation"], color="#d8f3dc"))
task_r_results.to_csv(TABLE_DIR / "task_r_all_metrics.csv", index=False)
pd.concat(tuning_tables, ignore_index=True).to_csv(TABLE_DIR / "tuning_all_candidates_so_far.csv", index=False)

## 13. NGBoost predictive uncertainty

A point metric cannot tell whether a probabilistic regressor expresses appropriate uncertainty. For test match $i$, NGBoost predicts $Y_i\mid X_i\sim\mathcal N(\mu_i,\sigma_i^2)$. We therefore add:

- mean negative log-likelihood (NLL);
- empirical 50%, 80%, and 95% interval coverage;
- average interval width;
- an interval plot ordered by predicted mean.

Even a validation-NLL-selected model can be systematically over- or under-dispersed. We therefore estimate one positive multiplicative scale factor from **validation residuals only**:

$$c=\sqrt{\frac{1}{n_{val}}\sum_i\frac{(y_i-\mu_i)^2}{\sigma_i^2}},\qquad
\sigma_i^{cal}=c\sigma_i.$$

This is the maximum-likelihood scalar correction when means and relative scales are held fixed. The test set is never used to determine $c$. Both raw and calibrated test diagnostics are retained, so calibration cannot conceal the original distribution. Good coverage should be close to nominal, but 46 test matches still create substantial sampling uncertainty.

In [ ]:
ngb_margin_dist = pipeline_pred_dist(reg_models["NGBoost"], X_pre["test"])
ngb_margin_val_dist = pipeline_pred_dist(reg_models["NGBoost"], X_pre["validation"])
ngb_mu = np.asarray(ngb_margin_dist.params["loc"], dtype=float)
ngb_sigma_raw = np.maximum(np.asarray(ngb_margin_dist.params["scale"], dtype=float), 1e-8)
ngb_val_mu = np.asarray(ngb_margin_val_dist.params["loc"], dtype=float)
ngb_val_sigma_raw = np.maximum(
    np.asarray(ngb_margin_val_dist.params["scale"], dtype=float), 1e-8
)

standardized_val_residual = (y_pre_r["validation"] - ngb_val_mu) / ngb_val_sigma_raw
ngb_sigma_scale = float(np.sqrt(np.mean(standardized_val_residual ** 2)))
assert np.isfinite(ngb_sigma_scale) and ngb_sigma_scale > 0

ngb_val_sigma = ngb_val_sigma_raw * ngb_sigma_scale
ngb_sigma = ngb_sigma_raw * ngb_sigma_scale
ngb_nll_raw = float(-np.mean(ngb_margin_dist.logpdf(y_pre_r["test"])))
ngb_nll = float(-np.mean(norm.logpdf(y_pre_r["test"], loc=ngb_mu, scale=ngb_sigma)))
ngb_val_nll_raw = float(-np.mean(ngb_margin_val_dist.logpdf(y_pre_r["validation"])))
ngb_val_nll = float(-np.mean(norm.logpdf(
    y_pre_r["validation"], loc=ngb_val_mu, scale=ngb_val_sigma
)))

coverage_rows = []
for scenario, sigma in [("Raw", ngb_sigma_raw), ("Validation-scale-calibrated", ngb_sigma)]:
    for level in [0.50, 0.80, 0.95]:
        z = norm.ppf((1 + level) / 2)
        lower, upper = ngb_mu - z * sigma, ngb_mu + z * sigma
        coverage_rows.append({
            "Scenario": scenario,
            "NominalCoverage": level,
            "EmpiricalCoverage": np.mean(
                (y_pre_r["test"] >= lower) & (y_pre_r["test"] <= upper)
            ),
            "MeanWidth": np.mean(upper - lower),
        })
coverage_table = pd.DataFrame(coverage_rows)
scale_calibration_table = pd.DataFrame([{
    "ScaleFactorFromValidation": ngb_sigma_scale,
    "ValidationNLLRaw": ngb_val_nll_raw,
    "ValidationNLLCalibrated": ngb_val_nll,
    "TestNLLRaw": ngb_nll_raw,
    "TestNLLCalibrated": ngb_nll,
    "ValidationMinSigmaRaw": float(np.min(ngb_val_sigma_raw)),
    "ValidationMedianSigmaRaw": float(np.median(ngb_val_sigma_raw)),
    "TestMinSigmaRaw": float(np.min(ngb_sigma_raw)),
    "TestMedianSigmaRaw": float(np.median(ngb_sigma_raw)),
}])
print(f"NGBoost validation-derived scale factor: {ngb_sigma_scale:.4f}")
print(f"NGBoost test NLL: raw={ngb_nll_raw:.4f}, calibrated={ngb_nll:.4f}")
display(scale_calibration_table.style.format(precision=4))
display(coverage_table.style.format(precision=4))

order = np.argsort(ngb_mu)
z80 = norm.ppf(0.90)
plt.figure(figsize=(13, 5))
plt.errorbar(np.arange(len(order)), ngb_mu[order], yerr=z80 * ngb_sigma[order],
             fmt="o", ms=4, capsize=2,
             label="NGBoost mean ± calibrated 80% interval")
plt.scatter(np.arange(len(order)), y_pre_r["test"][order], c="black", marker="x", label="Observed margin")
plt.axhline(0, color="grey", lw=1)
plt.xlabel("Test matches ordered by predicted margin")
plt.ylabel("Home − away final goals (clipped)")
plt.title("Task R — NGBoost validation-scale-calibrated uncertainty")
plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / "task_r_ngboost_intervals.png", dpi=180, bbox_inches="tight")
plt.show()

coverage_table.to_csv(TABLE_DIR / "task_r_ngboost_coverage.csv", index=False)
scale_calibration_table.to_csv(TABLE_DIR / "task_r_ngboost_scale_calibration.csv", index=False)

## 14. Can a margin distribution produce useful 1X2 probabilities?

The project asks whether Task R can be converted into Task C. With a continuous margin model, we approximate a draw by the interval $[-0.5,0.5]$ around zero:

$$P(A)=\Phi\!\left(\frac{-0.5-\mu}{\sigma}\right),\quad
P(D)=\Phi\!\left(\frac{0.5-\mu}{\sigma}\right)-P(A),\quad
P(H)=1-\Phi\!\left(\frac{0.5-\mu}{\sigma}\right).$$

NGBoost supplies a match-specific $\sigma_i$. For ordinary regressors, the code uses a single residual standard deviation estimated on validation data. This comparison directly tests whether distributional regression can replace a dedicated classifier.

In [ ]:
def normal_margin_to_proba(mu, sigma):
    mu = np.asarray(mu, dtype=float)
    sigma = np.maximum(np.asarray(sigma, dtype=float), 1e-4)
    p_away = norm.cdf((-0.5 - mu) / sigma)
    p_draw = norm.cdf((0.5 - mu) / sigma) - p_away
    p_home = 1.0 - p_away - p_draw
    return normalize_proba(np.column_stack([p_away, p_draw, p_home]))

margin_probability_rows = []
margin_test_probs = {}
for model_name, test_mu in reg_test_pred.items():
    if model_name == "NGBoost":
        sigma = ngb_sigma
    else:
        residual_sigma = np.std(y_pre_r["validation"] - reg_val_pred[model_name], ddof=1)
        sigma = np.repeat(max(residual_sigma, 1e-4), len(test_mu))
    p = normal_margin_to_proba(test_mu, sigma)
    margin_test_probs[model_name] = p
    margin_probability_rows.append({"Margin model": model_name,
                                    **classification_metrics(y_pre_c["test"], p)})

margin_to_class_results = pd.DataFrame(margin_probability_rows).sort_values("RPS")
direct_ngboost = {"Margin model": "Direct categorical NGBoost",
                  **classification_metrics(y_pre_c["test"], cls_test_cal["NGBoost"])}
market_reference = {"Margin model": "De-vigged market",
                    **classification_metrics(y_pre_c["test"], market_test)}
margin_to_class_results = pd.concat(
    [margin_to_class_results, pd.DataFrame([direct_ngboost, market_reference])], ignore_index=True
).sort_values("RPS")
display(margin_to_class_results.style.format(precision=4))
margin_to_class_results.to_csv(TABLE_DIR / "task_r_margin_to_outcome_probabilities.csv", index=False)

## 15. Ten worst margin predictions

The post-mortem table ranks the final NGBoost point forecast by absolute error and includes its 80% interval. A large error outside the interval is both inaccurate and overconfident; a large error inside a wide interval is inaccurate but probabilistically less surprising.

In [ ]:
z80 = norm.ppf(0.90)
worst_r = pre_split["test"][[
    "match_id", "kickoff", "home_team_name", "away_team_name", "label_margin"
]].copy()
worst_r["predicted_margin"] = ngb_mu
worst_r["predicted_sigma"] = ngb_sigma
worst_r["lower_80"] = ngb_mu - z80 * ngb_sigma
worst_r["upper_80"] = ngb_mu + z80 * ngb_sigma
worst_r["absolute_error"] = np.abs(worst_r["label_margin"] - worst_r["predicted_margin"])
worst_r["inside_80"] = worst_r["label_margin"].between(worst_r["lower_80"], worst_r["upper_80"])
worst_r = worst_r.nlargest(10, "absolute_error")
display(worst_r)
worst_r.to_csv(TABLE_DIR / "task_r_ten_worst_ngboost.csv", index=False)

## 16. Paper-faithful multivariate Gaussian NGBoost

This is the closest reproduction of O'Malley et al. (2023). The response is two-dimensional: full-time home goals and away goals. The paper's key hypothesis is that modeling conditional covariance improves joint probabilistic forecasts when the outputs remain dependent after conditioning on features.

We report:

- home- and away-goal RMSE;
- joint multivariate Normal NLL;
- multivariate Energy Score;
- predicted home/away goal correlation;
- marginal interval coverage for each goal output;
- joint elliptical coverage using the $\chi^2_2$ Mahalanobis threshold;
- 1X2 RPS after fixed-seed Monte Carlo discretization.

The Energy Score is the multivariate analogue of CRPS:

$$ES(F,y)=\mathbb E_F\|X-y\|-\tfrac12\mathbb E_F\|X-X'\|.$$

In [ ]:
def energy_score_mc(distribution, observed, n_samples=500, seed=SEED):
    rng = np.random.default_rng(seed)
    a = distribution.rvs(size=n_samples, random_state=rng)
    b = distribution.rvs(size=n_samples, random_state=rng)
    return float(np.linalg.norm(a - observed, axis=1).mean()
                 - 0.5 * np.linalg.norm(a - b, axis=1).mean())

def mvn_goal_probabilities(distributions, n_samples=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    for distribution in distributions:
        draws = distribution.rvs(size=n_samples, random_state=rng)
        scores = np.maximum(0, np.rint(draws)).astype(int)
        margin = scores[:, 0] - scores[:, 1]
        rows.append([np.mean(margin < 0), np.mean(margin == 0), np.mean(margin > 0)])
    return normalize_proba(np.asarray(rows))

if RUN_MULTIVARIATE:
    mv_prep = SimpleImputer(strategy="median", add_indicator=True)
    X_mv_train = mv_prep.fit_transform(X_pre["train"])
    X_mv_test = mv_prep.transform(X_pre["test"])
    y_mv_train = pre_split["train"][["home_score", "away_score"]].to_numpy(float)
    y_mv_test = pre_split["test"][["home_score", "away_score"]].to_numpy(float)

    mv_model = NGBRegressor(
        Dist=MultivariateNormal(2), Score=LogScore,
        Base=DecisionTreeRegressor(max_depth=2, min_samples_leaf=10, random_state=SEED),
        n_estimators=500 if RUN_MODE != "quick" else 250,
        learning_rate=0.03, minibatch_frac=0.8, col_sample=0.8,
        natural_gradient=True, random_state=SEED, verbose=False,
    )
    mv_model, mv_resources = fit_with_resources(mv_model, X_mv_train, y_mv_train)
    mv_dist = mv_model.pred_dist(X_mv_test)
    mv_scipy = list(mv_dist.scipy_distribution())
    mv_means = np.vstack([d.mean for d in mv_scipy])
    mv_covs = np.stack([d.cov for d in mv_scipy])
    mv_correlations = mv_covs[:, 0, 1] / np.sqrt(mv_covs[:, 0, 0] * mv_covs[:, 1, 1])
    mv_eigenvalues = np.linalg.eigvalsh(mv_covs)
    mv_nll = float(np.mean([-d.logpdf(y) for d, y in zip(mv_scipy, y_mv_test)]))
    mv_energy = float(np.mean([energy_score_mc(d, y, seed=SEED + i)
                               for i, (d, y) in enumerate(zip(mv_scipy, y_mv_test))]))
    mv_probs = mvn_goal_probabilities(mv_scipy)
    mv_delta = y_mv_test - mv_means
    mv_inverse_covs = np.linalg.inv(mv_covs)
    mv_mahalanobis_sq = np.einsum("ni,nij,nj->n", mv_delta, mv_inverse_covs, mv_delta)
    mv_marginal_sigma = np.sqrt(np.diagonal(mv_covs, axis1=1, axis2=2))
    mv_coverage_rows = []
    for level in [0.50, 0.80, 0.95]:
        z = norm.ppf((1 + level) / 2)
        lower = mv_means - z * mv_marginal_sigma
        upper = mv_means + z * mv_marginal_sigma
        mv_coverage_rows.append({
            "NominalCoverage": level,
            "HomeMarginalCoverage": float(np.mean(
                (y_mv_test[:, 0] >= lower[:, 0]) & (y_mv_test[:, 0] <= upper[:, 0])
            )),
            "AwayMarginalCoverage": float(np.mean(
                (y_mv_test[:, 1] >= lower[:, 1]) & (y_mv_test[:, 1] <= upper[:, 1])
            )),
            "JointEllipticalCoverage": float(np.mean(
                mv_mahalanobis_sq <= chi2.ppf(level, df=2)
            )),
            "MeanHomeIntervalWidth": float(np.mean(upper[:, 0] - lower[:, 0])),
            "MeanAwayIntervalWidth": float(np.mean(upper[:, 1] - lower[:, 1])),
        })
    mv_coverage = pd.DataFrame(mv_coverage_rows)
    mv_metrics = {
        "HomeGoalsRMSE": mean_squared_error(y_mv_test[:, 0], mv_means[:, 0]) ** 0.5,
        "AwayGoalsRMSE": mean_squared_error(y_mv_test[:, 1], mv_means[:, 1]) ** 0.5,
        "JointNLL": mv_nll,
        "EnergyScore": mv_energy,
        "MeanPredictedCorrelation": mv_correlations.mean(),
        "MinCovarianceEigenvalue": float(np.min(mv_eigenvalues)),
        "MedianCovarianceEigenvalue": float(np.median(mv_eigenvalues)),
        **{f"Outcome_{k}": v for k, v in classification_metrics(y_pre_c["test"], mv_probs).items()},
    }
    display(pd.DataFrame([mv_metrics]).T.rename(columns={0: "Value"}).style.format(precision=4))
    display(Markdown("### Multivariate marginal and joint coverage"))
    display(mv_coverage.style.format(precision=4))
    compute_rows.append({
        "Task": "P2-Multivariate", "Model": "MV Gaussian NGBoost", "Candidate": 1,
        "TrainSeconds": mv_resources.seconds, "PeakRSSMB": mv_resources.peak_rss_mb,
        "PeakDeltaMB": mv_resources.peak_delta_mb, "Selected": True,
        "Parameters": repr(mv_model.get_params()),
    })
    pd.DataFrame([mv_metrics]).to_csv(TABLE_DIR / "p2_multivariate_ngboost_metrics.csv", index=False)
    mv_coverage.to_csv(TABLE_DIR / "p2_multivariate_ngboost_coverage.csv", index=False)

    plt.figure(figsize=(7, 5))
    sns.histplot(mv_correlations, bins=15, kde=True)
    plt.axvline(mv_correlations.mean(), color="black", ls="--", label="mean")
    plt.xlabel("Predicted Corr(home goals, away goals)")
    plt.title("P2 reproduction — heterogeneous conditional correlation")
    plt.legend(); plt.tight_layout()
    plt.savefig(FIG_DIR / "p2_mvn_predicted_correlation.png", dpi=180, bbox_inches="tight")
    plt.show()
else:
    print("RUN_MULTIVARIATE=False: paper-faithful experiment intentionally skipped.")

## 17. Exact-kernel scaling experiment

An exact RBF kernel stores an $n\times n$ Gram matrix: space is $O(n^2)$ and solving can approach $O(n^3)$. Nystroem replaces it with an $n\times m$ feature map, approximately $O(nm)$ space for $m\ll n$.

Our pre-match dataset is small, so the absolute times are modest; the curve and theoretical memory column demonstrate the growth mechanism. The P1-eligible snapshot table has 7,854 rows overall. The code below reports the exact dense-Gram requirement for its training portion rather than relying on a hard-coded estimate.

In [ ]:
kernel_rows = []
max_n = len(X_pre["train"])
sizes = sorted(set([min(n, max_n) for n in [40, 80, 120, 180, 251]]))
for n in sizes:
    Xn, yn_c, yn_r = X_pre["train"].iloc[:n], y_pre_c["train"][:n], y_pre_r["train"][:n]
    experiments = {
        "Exact SVC": kernel_pipe(SVC(C=1.0, gamma="scale", probability=False, random_state=SEED)),
        "Exact Kernel Ridge": kernel_pipe(KernelRidge(alpha=1.0, kernel="rbf", gamma=0.01)),
        "Nystroem + Ridge": Pipeline([
            ("prep", Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)),
                               ("scaler", StandardScaler())])),
            ("nystroem", Nystroem(kernel="rbf", gamma=0.01, n_components=min(80, n), random_state=SEED)),
            ("model", Ridge(alpha=1.0)),
        ]),
    }
    for name, estimator in experiments.items():
        target = yn_c if name == "Exact SVC" else yn_r
        _, resources = fit_with_resources(estimator, Xn, target)
        kernel_rows.append({"Method": name, "n": n, "TrainSeconds": resources.seconds,
                            "PeakDeltaMB": resources.peak_delta_mb,
                            "DenseGramTheoreticalMB": n * n * 8 / 2**20})

kernel_scaling = pd.DataFrame(kernel_rows)
display(kernel_scaling.style.format(precision=4))
plt.figure(figsize=(8, 5))
sns.lineplot(data=kernel_scaling, x="n", y="TrainSeconds", hue="Method", marker="o")
plt.title("Kernel scaling on chronological training prefixes")
plt.ylabel("Fit time (seconds)")
plt.tight_layout()
plt.savefig(FIG_DIR / "kernel_scaling_time.png", dpi=180, bbox_inches="tight")
plt.show()

snapshot_n = len(X_live["train"])
print(f"A dense float64 Gram matrix for {snapshot_n:,} training snapshots is "
      f"{snapshot_n**2 * 8 / 2**30:.2f} GiB before solver overhead.")
kernel_scaling.to_csv(TABLE_DIR / "kernel_scaling.csv", index=False)

# Part III — Task L: in-play forecasts

Each live row combines the validation-frozen 146-feature pre-match vector with match events observed at its snapshot boundary. The 18 Berrar fields are constant within a match; only the audited event-prefix fields change over time. All 21 snapshots inherit the parent match split. The same tree families are retrained on this larger table; SVM and Kernel Ridge are omitted because the brief assigns them only to Tasks C/R.

Classification models are calibrated on validation snapshots. This does not split matches because the full validation set is already match-disjoint. Regression models predict updated final margin. Resource measurements are recorded for the compute comparison.

In [ ]:
live_cls_models = {}
live_cls_calibrators = {}
live_cls_val_raw = {}
live_cls_test_raw = {}
live_cls_test_cal = {}

for model_name, candidates in classification_candidates(live=True).items():
    print(f"Fitting Task L classification: {model_name}")
    fitted, tuning, selected = select_candidate(
        model_name, candidates,
        X_live["train"], y_live_c["train"], X_live["validation"], y_live_c["validation"],
        task="classification",
    )
    tuning.insert(0, "Task", "L-classification")
    tuning_tables.append(tuning)
    compute_rows.append({"Task": "L-classification", **selected})
    live_cls_models[model_name] = fitted
    val_raw = align_proba(fitted, X_live["validation"])
    test_raw = align_proba(fitted, X_live["test"])
    calibrator = MulticlassPlattCalibrator().fit(val_raw, y_live_c["validation"])
    live_cls_calibrators[model_name] = calibrator
    live_cls_val_raw[model_name] = val_raw
    live_cls_test_raw[model_name] = test_raw
    live_cls_test_cal[model_name] = calibrator.predict_proba(test_raw)
    if SAVE_MODELS:
        joblib.dump({"model": fitted, "calibrator": calibrator},
                    MODEL_DIR / f"task_l_classifier_{model_name.lower().replace(' ', '_')}.joblib")

live_reg_models = {}
live_reg_val_pred = {}
live_reg_test_pred = {}
for model_name, candidates in regression_candidates(live=True).items():
    print(f"Fitting Task L regression: {model_name}")
    fitted, tuning, selected = select_candidate(
        model_name, candidates,
        X_live["train"], y_live_r["train"], X_live["validation"], y_live_r["validation"],
        task="regression",
    )
    tuning.insert(0, "Task", "L-regression")
    tuning_tables.append(tuning)
    compute_rows.append({"Task": "L-regression", **selected})
    live_reg_models[model_name] = fitted
    live_reg_val_pred[model_name] = np.asarray(fitted.predict(X_live["validation"]), dtype=float)
    live_reg_test_pred[model_name] = np.asarray(fitted.predict(X_live["test"]), dtype=float)
    if SAVE_MODELS:
        joblib.dump(fitted, MODEL_DIR / f"task_l_regressor_{model_name.lower().replace(' ', '_')}.joblib")

pd.concat(tuning_tables, ignore_index=True).to_csv(TABLE_DIR / "tuning_all_candidates.csv", index=False)
print("All Task L models fitted and individually checkpointed.")

## 18. Frozen pre-match references and aggregate live results

For each live model family, the fair reference is its **own** calibrated pre-match prediction copied to every snapshot of that match. A useful live model should reduce RPS and margin error relative to that frozen forecast as events arrive.

The headline result tables include one NGBoost frozen reference for orientation. The family-matched delta table below is stricter: positive `RPS_improvement` and `RMSE_improvement` mean live information helped.

In [ ]:
pre_test_match_ids = pre_split["test"]["match_id"].to_numpy()
live_test_match_ids = live_split["test"]["match_id"].to_numpy()

def repeat_match_values(match_ids_source, values, match_ids_target):
    mapping = {int(match_id): value for match_id, value in zip(match_ids_source, values)}
    return np.asarray([mapping[int(match_id)] for match_id in match_ids_target])

frozen_cls = {}
frozen_reg = {}
for model_name in live_cls_models:
    frozen_cls[model_name] = repeat_match_values(
        pre_test_match_ids, cls_test_cal[model_name], live_test_match_ids
    )
for model_name in live_reg_models:
    frozen_reg[model_name] = repeat_match_values(
        pre_test_match_ids, reg_test_pred[model_name], live_test_match_ids
    )

task_l_cls_rows = []
for name in live_cls_models:
    task_l_cls_rows.append({"Model": name, "Calibration": "Raw",
                            **classification_metrics(y_live_c["test"], live_cls_test_raw[name])})
    task_l_cls_rows.append({"Model": name, "Calibration": "Platt",
                            **classification_metrics(y_live_c["test"], live_cls_test_cal[name])})
task_l_cls_rows.append({
    "Model": "Frozen pre-match NGBoost", "Calibration": "Platt pre-match",
    **classification_metrics(y_live_c["test"], frozen_cls["NGBoost"]),
})
task_l_cls_results = pd.DataFrame(task_l_cls_rows).sort_values("RPS")

task_l_reg_rows = [
    {"Model": name, **regression_metrics(y_live_r["test"], pred)}
    for name, pred in live_reg_test_pred.items()
]
task_l_reg_rows.append({
    "Model": "Frozen pre-match NGBoost",
    **regression_metrics(y_live_r["test"], frozen_reg["NGBoost"]),
})
task_l_reg_results = pd.DataFrame(task_l_reg_rows).sort_values("RMSE")

family_delta = []
for name in live_cls_models:
    live_m = classification_metrics(y_live_c["test"], live_cls_test_cal[name])
    frozen_m = classification_metrics(y_live_c["test"], frozen_cls[name])
    reg_live_m = regression_metrics(y_live_r["test"], live_reg_test_pred[name])
    reg_frozen_m = regression_metrics(y_live_r["test"], frozen_reg[name])
    family_delta.append({
        "Model": name,
        "RPS_live": live_m["RPS"], "RPS_frozen": frozen_m["RPS"],
        "RPS_improvement": frozen_m["RPS"] - live_m["RPS"],
        "RMSE_live": reg_live_m["RMSE"], "RMSE_frozen": reg_frozen_m["RMSE"],
        "RMSE_improvement": reg_frozen_m["RMSE"] - reg_live_m["RMSE"],
    })
family_delta = pd.DataFrame(family_delta).sort_values("RPS_improvement", ascending=False)

display(Markdown("### Task L classification — all metrics"))
display(task_l_cls_results.style.format(precision=4))
display(Markdown("### Task L regression — all metrics"))
display(task_l_reg_results.style.format(precision=4))
display(Markdown("### Family-matched live improvement over frozen pre-match forecasts"))
display(family_delta.style.format(precision=4).background_gradient(
    subset=["RPS_improvement", "RMSE_improvement"], cmap="RdYlGn"
))

task_l_cls_results.to_csv(TABLE_DIR / "task_l_classification_all_metrics.csv", index=False)
task_l_reg_results.to_csv(TABLE_DIR / "task_l_regression_all_metrics.csv", index=False)
family_delta.to_csv(TABLE_DIR / "task_l_family_matched_improvement.csv", index=False)

## 19. Every live metric as a function of match time

Aggregate live scores overrepresent long periods and can hide when the stream becomes useful. The next cell evaluates each snapshot boundary separately. It writes the complete numeric tables and plots:

- classification: RPS, log-loss, Brier, ECE, accuracy, macro precision, macro recall, macro F1;
- regression: MAE, RMSE, correlation.

The black dashed curve is the frozen pre-match NGBoost reference. It is flat by construction. The expected pattern is decreasing proper scores/errors and increasing accuracy/correlation, especially after goals or red cards. If a curve does not improve, that is an honest negative result.

In [ ]:
snapshot_meta = live_split["test"][["snapshot_id", "snapshot_rank"]].reset_index(drop=True)

minute_cls_rows = []
for model_name, probs in {**live_cls_test_cal, "Frozen pre-match NGBoost": frozen_cls["NGBoost"]}.items():
    for rank in sorted(snapshot_meta["snapshot_rank"].unique()):
        mask = snapshot_meta["snapshot_rank"].eq(rank).to_numpy()
        snapshot_id = snapshot_meta.loc[mask, "snapshot_id"].iloc[0]
        minute_cls_rows.append({"Model": model_name, "snapshot_id": snapshot_id, "snapshot_rank": rank,
                                **classification_metrics(y_live_c["test"][mask], probs[mask])})
minute_cls = pd.DataFrame(minute_cls_rows)

minute_reg_rows = []
for model_name, pred in {**live_reg_test_pred, "Frozen pre-match NGBoost": frozen_reg["NGBoost"]}.items():
    for rank in sorted(snapshot_meta["snapshot_rank"].unique()):
        mask = snapshot_meta["snapshot_rank"].eq(rank).to_numpy()
        snapshot_id = snapshot_meta.loc[mask, "snapshot_id"].iloc[0]
        minute_reg_rows.append({"Model": model_name, "snapshot_id": snapshot_id, "snapshot_rank": rank,
                                **regression_metrics(y_live_r["test"][mask], pred[mask])})
minute_reg = pd.DataFrame(minute_reg_rows)

cls_metric_names = ["RPS", "LogLoss", "Brier", "ECE", "Accuracy",
                    "MacroPrecision", "MacroRecall", "MacroF1"]
fig, axes = plt.subplots(2, 4, figsize=(22, 10), sharex=True)
for ax, metric in zip(axes.flat, cls_metric_names):
    sns.lineplot(data=minute_cls, x="snapshot_rank", y=metric, hue="Model", ax=ax,
                 legend=(metric == "RPS"), estimator=None)
    ax.set_title(metric); ax.set_xlabel("Snapshot minute/rank")
    if metric != "RPS" and ax.legend_ is not None:
        ax.legend_.remove()
axes.flat[0].legend(bbox_to_anchor=(0, 1.02), loc="lower left", ncol=2, fontsize=8)
fig.suptitle("Task L classification metrics versus match time", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "task_l_classification_metrics_by_minute.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
for ax, metric in zip(axes, ["MAE", "RMSE", "Correlation"]):
    sns.lineplot(data=minute_reg, x="snapshot_rank", y=metric, hue="Model", ax=ax,
                 legend=(metric == "MAE"), estimator=None)
    ax.set_title(metric); ax.set_xlabel("Snapshot minute/rank")
    if metric != "MAE" and ax.legend_ is not None:
        ax.legend_.remove()
axes[0].legend(bbox_to_anchor=(0, 1.02), loc="lower left", ncol=2, fontsize=8)
fig.suptitle("Task L final-margin metrics versus match time", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "task_l_regression_metrics_by_minute.png", dpi=180, bbox_inches="tight")
plt.show()

minute_cls.to_csv(TABLE_DIR / "task_l_classification_by_minute.csv", index=False)
minute_reg.to_csv(TABLE_DIR / "task_l_regression_by_minute.csv", index=False)
display(minute_cls.pivot(index="snapshot_id", columns="Model", values="RPS").style.format(precision=3))

## 20. Calibration by game phase

The brief requires calibration for 0–15, 15–30, …, 75–90 minutes. Half-time and full-time boundary rows are assigned using their numeric rank. The table reports ECE, RPS, log-loss, and sample count for every live classifier in every phase; the NGBoost diagrams then show class-wise reliability within each phase.

Phase-level test groups contain only a few hundred correlated snapshots from 46 matches, so treat small ECE differences cautiously.

In [ ]:
phase_edges = [-0.1, 15, 30, 45.75, 60, 75, 90.6]
phase_labels = ["0–15", "15–30", "30–45/HT", "HT–60", "60–75", "75–FT"]
test_phases = pd.cut(snapshot_meta["snapshot_rank"], bins=phase_edges, labels=phase_labels,
                     include_lowest=True, right=True)

phase_rows = []
for model_name, probs in live_cls_test_cal.items():
    for phase in phase_labels:
        mask = test_phases.eq(phase).to_numpy()
        metrics = classification_metrics(y_live_c["test"][mask], probs[mask])
        phase_rows.append({"Model": model_name, "Phase": phase, "Rows": mask.sum(),
                           "Matches": live_split["test"].loc[mask, "match_id"].nunique(),
                           "ECE": metrics["ECE"], "RPS": metrics["RPS"],
                           "LogLoss": metrics["LogLoss"], "Brier": metrics["Brier"]})
phase_calibration = pd.DataFrame(phase_rows)
display(phase_calibration.pivot(index="Phase", columns="Model", values="ECE").style.format(precision=3))

for phase in phase_labels:
    mask = test_phases.eq(phase).to_numpy()
    plot_reliability(
        y_live_c["test"][mask], live_cls_test_raw["NGBoost"][mask], live_cls_test_cal["NGBoost"][mask],
        f"Task L NGBoost reliability — phase {phase}",
        f"task_l_ngboost_reliability_phase_{phase.replace('–', '_').replace('/', '_')}.png",
    )
phase_calibration.to_csv(TABLE_DIR / "task_l_phase_calibration.csv", index=False)

## 21. Live feature-group ablation

This experiment measures information value rather than only ranking individual features. The stages are nested:

1. pre-match history only, including the frozen 18-feature P1 context;
2. add clock and score state;
3. add man-advantage/red-card information;
4. add cumulative live event counts;
5. add recent-window momentum (the full live vector).

LightGBM is held fixed across stages. Improvements can therefore be attributed to new information groups rather than a different learner or tuning budget.

In [ ]:
pre_context = [c for c in live_features if c in pre_features]
clock_score = [c for c in live_features if c in {
    "snapshot_minute", "snapshot_rank", "snapshot_period", "reference_second",
    "n_events_so_far", "current_home_score", "current_away_score",
    "current_goal_difference", "time_remaining_minutes"
}]
man_advantage = [c for c in live_features if "red_card" in c or c == "man_advantage_home"]
recent = [c for c in live_features if "last5" in c or "last10" in c]
cumulative = [c for c in live_features
              if c not in set(pre_context + clock_score + man_advantage + recent)]

ablation_sets = {
    "1 Pre-match context": pre_context,
    "2 + clock/score": list(dict.fromkeys(pre_context + clock_score)),
    "3 + man advantage": list(dict.fromkeys(pre_context + clock_score + man_advantage)),
    "4 + cumulative stream": list(dict.fromkeys(pre_context + clock_score + man_advantage + cumulative)),
    "5 + recent momentum (full)": live_features,
}

ablation_rows = []
ablation_models = {}
for stage, columns in ablation_sets.items():
    classifier = tree_pipe(LGBMClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=4, num_leaves=15,
        min_child_samples=15, subsample=0.8, colsample_bytree=0.8, reg_lambda=2,
        random_state=SEED, n_jobs=N_JOBS, verbosity=-1,
    ))
    regressor = tree_pipe(LGBMRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=4, num_leaves=15,
        min_child_samples=15, subsample=0.8, colsample_bytree=0.8, reg_lambda=2,
        random_state=SEED, n_jobs=N_JOBS, verbosity=-1,
    ))
    classifier.fit(X_live["train"][columns], y_live_c["train"])
    raw_val = align_proba(classifier, X_live["validation"][columns])
    raw_test = align_proba(classifier, X_live["test"][columns])
    calibrator = MulticlassPlattCalibrator().fit(raw_val, y_live_c["validation"])
    probs = calibrator.predict_proba(raw_test)
    regressor.fit(X_live["train"][columns], y_live_r["train"])
    margin = regressor.predict(X_live["test"][columns])
    ablation_models[stage] = (classifier, calibrator, regressor, columns)
    ablation_rows.append({"Stage": stage, "Features": len(columns),
                          **classification_metrics(y_live_c["test"], probs),
                          **{f"Margin_{k}": v for k, v in regression_metrics(y_live_r["test"], margin).items()}})

ablation_results = pd.DataFrame(ablation_rows)
display(ablation_results.style.format(precision=4))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(data=ablation_results, x="Stage", y="RPS", marker="o", ax=axes[0])
sns.lineplot(data=ablation_results, x="Stage", y="Margin_RMSE", marker="o", ax=axes[1])
for ax in axes:
    ax.tick_params(axis="x", rotation=25)
axes[0].set_title("Outcome information gain"); axes[1].set_title("Margin information gain")
fig.tight_layout()
fig.savefig(FIG_DIR / "task_l_feature_group_ablation.png", dpi=180, bbox_inches="tight")
plt.show()
ablation_results.to_csv(TABLE_DIR / "task_l_feature_group_ablation.csv", index=False)

## 22. Ten worst live forecasts for each output

The classification table uses per-snapshot log-loss; the regression table uses absolute margin error. Snapshot time is retained so the report can distinguish a defensible early error from an implausibly confident late error. The same rows become the candidates for local SHAP explanations.

In [ ]:
live_ngb_probs = live_cls_test_cal["NGBoost"]
live_ngb_margin = live_reg_test_pred["NGBoost"]

worst_lc = live_split["test"][["match_id", "snapshot_id", "snapshot_rank", "label_outcome"]].copy()
worst_lc[["p_away", "p_draw", "p_home"]] = live_ngb_probs
worst_lc["predicted"] = [INT_TO_CLASS[i] for i in live_ngb_probs.argmax(axis=1)]
worst_lc["case_log_loss"] = -np.log(np.clip(
    live_ngb_probs[np.arange(len(live_ngb_probs)), y_live_c["test"]], 1e-12, 1.0
))
worst_lc = worst_lc.nlargest(10, "case_log_loss")

worst_lr = live_split["test"][["match_id", "snapshot_id", "snapshot_rank", "label_margin"]].copy()
worst_lr["predicted_margin"] = live_ngb_margin
worst_lr["absolute_error"] = np.abs(worst_lr["label_margin"] - worst_lr["predicted_margin"])
worst_lr = worst_lr.nlargest(10, "absolute_error")

display(Markdown("### Ten worst live outcome snapshots")); display(worst_lc)
display(Markdown("### Ten worst live margin snapshots")); display(worst_lr)
worst_lc.to_csv(TABLE_DIR / "task_l_ten_worst_outcome_ngboost.csv", index=False)
worst_lr.to_csv(TABLE_DIR / "task_l_ten_worst_margin_ngboost.csv", index=False)

# Part IV — Explainability with SHAP

SHAP is applied to every tree family required by the brief—Random Forest, GBM, XGBoost, and LightGBM—for Tasks C, R, and both Task L outputs.

- TreeExplainer is attempted first.
- If a particular sklearn multiclass ensemble is unsupported by TreeExplainer, the code falls back to model-agnostic permutation SHAP on a smaller evaluation sample and records that fallback.
- Classification beeswarms explain the **Home-win raw model output**. Regression beeswarms explain predicted margin.
- Calibrators are not included in the explanation, so SHAP additivity refers to the underlying model score/probability, not the post-hoc Platt mapping.

NGBoost contains separate tree ensembles for each distribution parameter, which is not a standard single TreeSHAP object. Its uncertainty is interpreted through predicted $\mu$/$\sigma$ and fidelity analysis; the required TreeSHAP plots use the four directly supported tree baselines.

In [ ]:
TREE_MODEL_NAMES = ["Random Forest", "GBM", "XGBoost", "LightGBM"]
SHAP_SAMPLE = {"quick": 30, "standard": 80, "full": 150}[RUN_MODE]
SHAP_BACKGROUND = {"quick": 15, "standard": 30, "full": 50}[RUN_MODE]

def transformed_feature_names(preprocessor, original_names, n_transformed):
    try:
        names = list(preprocessor.get_feature_names_out(original_names))
    except Exception:
        names = list(original_names)
    if len(names) != n_transformed:
        names = [f"feature_{i}" for i in range(n_transformed)]
    return names

def select_shap_output(explanation, class_id=2):
    values = np.asarray(explanation.values)
    if values.ndim == 2:
        return explanation
    if values.ndim == 3 and values.shape[-1] > class_id:
        return explanation[:, :, class_id]
    if values.ndim == 3 and values.shape[1] > class_id:
        vals = values[:, class_id, :]
        return shap.Explanation(values=vals, base_values=np.asarray(explanation.base_values)[:, class_id],
                                data=explanation.data, feature_names=explanation.feature_names)
    raise ValueError(f"Unexpected SHAP shape: {values.shape}")

def build_shap_context(pipeline, X, original_names, classification=False):
    prep = pipeline.named_steps["prep"]
    estimator = pipeline.named_steps["model"]
    Xt = np.asarray(prep.transform(X))
    names = transformed_feature_names(prep, original_names, Xt.shape[1])
    rng = np.random.default_rng(SEED)
    background_idx = rng.choice(len(Xt), size=min(SHAP_BACKGROUND, len(Xt)), replace=False)
    eval_idx = rng.choice(len(Xt), size=min(SHAP_SAMPLE, len(Xt)), replace=False)
    background = Xt[background_idx]
    method = "TreeExplainer"
    try:
        explainer = shap.TreeExplainer(estimator, data=background, feature_names=names)
        explanation = explainer(Xt[eval_idx])
    except Exception as exc:
        method = f"Permutation fallback ({type(exc).__name__})"
        eval_idx = eval_idx[:min(30, len(eval_idx))]
        prediction_function = estimator.predict_proba if classification else estimator.predict
        explainer = shap.Explainer(prediction_function, background, algorithm="permutation",
                                   feature_names=names, seed=SEED)
        explanation = explainer(Xt[eval_idx], max_evals=2 * Xt.shape[1] + 1)
    selected = select_shap_output(explanation, class_id=2) if classification else explanation
    return {"explainer": explainer, "explanation": explanation, "selected": selected,
            "Xt": Xt, "feature_names": names, "method": method, "eval_idx": eval_idx,
            "estimator": estimator, "classification": classification}

def plot_global_shap(context, task_name, model_name):
    shap.plots.beeswarm(context["selected"], max_display=18, show=False)
    plt.title(f"{task_name} — {model_name} SHAP summary")
    plt.tight_layout()
    safe = f"{task_name}_{model_name}".lower().replace(" ", "_").replace("—", "_")
    plt.savefig(FIG_DIR / f"shap_global_{safe}.png", dpi=180, bbox_inches="tight")
    plt.show()

    mean_abs = np.mean(np.abs(np.asarray(context["selected"].values)), axis=0)
    estimator = context["estimator"]
    built_in = getattr(estimator, "feature_importances_", np.repeat(np.nan, len(mean_abs)))
    table = pd.DataFrame({"Feature": context["feature_names"], "MeanAbsSHAP": mean_abs,
                          "BuiltinImportance": built_in})
    table["Task"] = task_name
    table["Model"] = model_name
    table["SHAPMethod"] = context["method"]
    return table.sort_values("MeanAbsSHAP", ascending=False)

shap_contexts = {}
shap_importance_tables = []
if RUN_SHAP:
    suites = [
        ("Task C", cls_models, X_pre["test"], pre_features, True),
        ("Task R", reg_models, X_pre["test"], pre_features, False),
        ("Task L outcome", live_cls_models, X_live["test"], live_features, True),
        ("Task L margin", live_reg_models, X_live["test"], live_features, False),
    ]
    for task_name, model_dict, X_eval, names, is_classification in suites:
        for model_name in TREE_MODEL_NAMES:
            print(f"SHAP: {task_name} / {model_name}")
            try:
                context = build_shap_context(model_dict[model_name], X_eval, names, is_classification)
                shap_contexts[(task_name, model_name)] = context
                table = plot_global_shap(context, task_name, model_name)
                shap_importance_tables.append(table)
                display(table.head(12).style.format({"MeanAbsSHAP": "{:.5f}",
                                                     "BuiltinImportance": "{:.5f}"}))
            except Exception as exc:
                print(f"SHAP failed for {task_name}/{model_name}: {type(exc).__name__}: {exc}")

    if shap_importance_tables:
        shap_importances = pd.concat(shap_importance_tables, ignore_index=True)
        shap_importances.to_csv(TABLE_DIR / "shap_global_importances_all_tree_models.csv", index=False)
else:
    print("RUN_SHAP=False: explanation cells intentionally skipped.")

## 23. Local SHAP explanations of confidently wrong predictions

The next cell explains the worst LightGBM prediction for each task because LightGBM provides direct TreeSHAP support. A waterfall begins at the model's expected output and adds signed feature contributions until it reaches the selected prediction.

These are model-specific explanations, not causal claims. A feature may receive high attribution because it proxies team strength or game state, not because changing it would cause the result to change.

In [ ]:
def plot_local_waterfall(context, row_index, title, save_name, class_id=2):
    explanation = context["explainer"](context["Xt"][[row_index]])
    if context["classification"]:
        explanation = select_shap_output(explanation, class_id=class_id)
    shap.plots.waterfall(explanation[0], max_display=18, show=False)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(FIG_DIR / save_name, dpi=180, bbox_inches="tight")
    plt.show()

if RUN_SHAP:
    # Worst LightGBM pre-match class forecast.
    p = cls_test_cal["LightGBM"]
    worst_idx_c = int(np.argmax(-np.log(np.clip(p[np.arange(len(p)), y_pre_c["test"]], 1e-12, 1))))
    plot_local_waterfall(shap_contexts[("Task C", "LightGBM")], worst_idx_c,
                         "Task C — worst LightGBM forecast (Home-output explanation)",
                         "shap_local_task_c_worst_lightgbm.png", class_id=2)

    worst_idx_r = int(np.argmax(np.abs(y_pre_r["test"] - reg_test_pred["LightGBM"])))
    plot_local_waterfall(shap_contexts[("Task R", "LightGBM")], worst_idx_r,
                         "Task R — worst LightGBM margin forecast",
                         "shap_local_task_r_worst_lightgbm.png")

    p_live = live_cls_test_cal["LightGBM"]
    worst_idx_lc = int(np.argmax(-np.log(np.clip(
        p_live[np.arange(len(p_live)), y_live_c["test"]], 1e-12, 1))))
    actual_class = int(y_live_c["test"][worst_idx_lc])
    plot_local_waterfall(shap_contexts[("Task L outcome", "LightGBM")], worst_idx_lc,
                         f"Task L — worst outcome snapshot (actual {CLASS_NAMES[actual_class]} output)",
                         "shap_local_task_l_outcome_worst_lightgbm.png", class_id=actual_class)

    worst_idx_lr = int(np.argmax(np.abs(y_live_r["test"] - live_reg_test_pred["LightGBM"])))
    plot_local_waterfall(shap_contexts[("Task L margin", "LightGBM")], worst_idx_lr,
                         "Task L — worst margin snapshot",
                         "shap_local_task_l_margin_worst_lightgbm.png")

## 24. Full-match SHAP timeline

We select the test match with the largest observed final goal total, explain all 21 LightGBM outcome snapshots, and track the six features with the largest mean absolute attribution for the match's actual final class.

This is the required end-to-end live explanation: pre-match form should matter early; current score, time remaining, and man advantage should take over when the event stream changes the state. Flat attributions are also informative—they indicate that the model is not using the live signal as intended.

In [ ]:
if RUN_SHAP:
    match_choice = (live_split["test"].groupby("match_id")
                    .agg(final_goals=("final_home_score", "max"),
                         final_away=("final_away_score", "max")))
    match_choice["total_goals"] = match_choice["final_goals"] + match_choice["final_away"]
    timeline_match_id = int(match_choice["total_goals"].idxmax())
    timeline_mask = live_split["test"]["match_id"].eq(timeline_match_id).to_numpy()
    timeline_indices = np.flatnonzero(timeline_mask)
    timeline_frame = live_split["test"].loc[timeline_mask].sort_values("snapshot_rank")
    timeline_indices = timeline_indices[np.argsort(live_split["test"].loc[timeline_mask, "snapshot_rank"].to_numpy())]
    actual_class = int(encode_outcome(timeline_frame["label_outcome"])[0])

    context = shap_contexts[("Task L outcome", "LightGBM")]
    explanation = context["explainer"](context["Xt"][timeline_indices])
    selected = select_shap_output(explanation, class_id=actual_class)
    values = np.asarray(selected.values)
    top_idx = np.argsort(np.mean(np.abs(values), axis=0))[-6:][::-1]
    timeline_shap = pd.DataFrame(values[:, top_idx], columns=np.asarray(context["feature_names"])[top_idx])
    timeline_shap["snapshot_rank"] = timeline_frame["snapshot_rank"].to_numpy()
    timeline_long = timeline_shap.melt("snapshot_rank", var_name="Feature", value_name="SHAP value")

    plt.figure(figsize=(14, 7))
    sns.lineplot(data=timeline_long, x="snapshot_rank", y="SHAP value", hue="Feature", marker="o")
    plt.axhline(0, color="black", lw=1)
    plt.title(f"Full-match SHAP timeline — match {timeline_match_id}, actual {CLASS_NAMES[actual_class]}")
    plt.xlabel("Snapshot minute/rank")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "shap_timeline_full_match_lightgbm.png", dpi=180, bbox_inches="tight")
    plt.show()
    timeline_long.to_csv(TABLE_DIR / "shap_timeline_full_match.csv", index=False)

# Part V — Compute, automatic findings, and reproducibility

## 25. Training time and peak memory

This table contains every selected model fit. Candidate-level timing remains in `tuning_all_candidates.csv`. Comparisons should be interpreted with care: Python process RSS includes the current notebook baseline, parallel native libraries may allocate briefly between samples, and NGBoost performs many sequential weak-learner fits. Still, the same instrumentation and runtime are used for all models, making family-level differences useful.

In [ ]:
compute_table = pd.DataFrame(compute_rows)
preferred_columns = [
    "Task", "Model", "Candidate", "SelectionCriterion", "TrainSeconds",
    "PeakRSSMB", "PeakDeltaMB", "validation_RPS", "validation_RMSE",
    "validation_NLL", "validation_MinSigma", "validation_MedianSigma",
    "validation_Coverage80Raw", "Selected",
]
display(compute_table[[c for c in preferred_columns if c in compute_table.columns]]
        .sort_values(["Task", "TrainSeconds"]).style.format(precision=4))

plt.figure(figsize=(12, 7))
sns.barplot(data=compute_table, y="Model", x="TrainSeconds", hue="Task")
plt.xscale("log")
plt.title("Selected-model training time (log scale)")
plt.tight_layout()
plt.savefig(FIG_DIR / "compute_training_time.png", dpi=180, bbox_inches="tight")
plt.show()

compute_table.to_csv(TABLE_DIR / "compute_and_peak_memory.csv", index=False)

## 26. Automatic results narrative

This cell converts the actual run—not expected behavior—into a compact set of report-ready statements. It names the best models, quantifies the gap to market, checks calibration direction, and measures NGBoost's improvement from kickoff to full time. Copy the numbers into the report, then add football-specific interpretation from the worst-case and SHAP tables.

In [ ]:
task_c_platt = task_c_results[
    task_c_results["Calibration"].isin(["Platt", "Market baseline"])
].copy()
best_c = task_c_platt.loc[task_c_platt["RPS"].idxmin()]
market_c = task_c_results[task_c_results["Model"].eq("De-vigged market")].iloc[0]
ngboost_c = task_c_results[(task_c_results["Model"].eq("NGBoost")) &
                           (task_c_results["Calibration"].eq("Platt"))].iloc[0]
best_r = task_r_results.loc[task_r_results["RMSE"].idxmin()]
task_l_platt = task_l_cls_results[
    task_l_cls_results["Calibration"].isin(["Platt", "Platt pre-match"])
]
best_lc = task_l_platt.loc[task_l_platt["RPS"].idxmin()]
best_lr = task_l_reg_results.loc[task_l_reg_results["RMSE"].idxmin()]

ngb_minute = minute_cls[minute_cls["Model"].eq("NGBoost")].sort_values("snapshot_rank")
ngb_frozen_minute = minute_cls[minute_cls["Model"].eq("Frozen pre-match NGBoost")].sort_values("snapshot_rank")
first_live, last_live = ngb_minute.iloc[0], ngb_minute.iloc[-1]
first_frozen, last_frozen = ngb_frozen_minute.iloc[0], ngb_frozen_minute.iloc[-1]

narrative = f'''
### Results generated by this run

- **P1 representation:** every downstream model used the validation-frozen **MD1 + P1 Home/Away** view ({len(pre_features)} pre-match features; {len(live_features)} live features after carrying the 18 P1 values into each snapshot). The separate upstream RF ablation remains the evidence for the 128-vs-146 feature decision.
- **Task C:** the lowest test RPS was **{best_c['Model']} ({best_c['RPS']:.4f})**. Calibrated NGBoost achieved **{ngboost_c['RPS']:.4f}**, versus **{market_c['RPS']:.4f}** for the de-vigged market; NGBoost's market gap was **{ngboost_c['RPS'] - market_c['RPS']:+.4f}** (negative means better than market).
- **Task R:** the lowest test RMSE was **{best_r['Model']} ({best_r['RMSE']:.4f})**, with MAE **{best_r['MAE']:.4f}** and correlation **{best_r['Correlation']:.4f}**.
- **Task L outcome:** the lowest aggregate snapshot RPS was **{best_lc['Model']} ({best_lc['RPS']:.4f})**.
- **Task L margin:** the lowest aggregate snapshot RMSE was **{best_lr['Model']} ({best_lr['RMSE']:.4f})**.
- **NGBoost information gain:** live RPS changed from **{first_live['RPS']:.4f}** at {first_live['snapshot_id']} to **{last_live['RPS']:.4f}** at {last_live['snapshot_id']}; the frozen reference stayed at **{first_frozen['RPS']:.4f} → {last_frozen['RPS']:.4f}**.
- **Uncertainty:** Task R NGBoost was selected by validation NLL. Its test NLL changed from **{ngb_nll_raw:.4f} raw** to **{ngb_nll:.4f} after validation-only scale calibration** (scale factor **{ngb_sigma_scale:.3f}**). Compare both scenarios in `task_r_ngboost_coverage.csv` before claiming well-calibrated intervals.

These statements describe this held-out season segment only. With 46 test matches from one competition-season, model rankings have high sampling uncertainty and should not be generalized to other leagues or eras without additional seasons.
'''
display(Markdown(narrative))
(OUTPUT_DIR / "automatic_results_narrative.md").write_text(narrative, encoding="utf-8")

## 27. Save predictions, models, versions, seeds, and git history

The final cell creates a self-contained evidence folder and ZIP. It includes:

- all numeric result tables and figures already written above;
- row-level test predictions for all three tasks;
- selected pre-match models and calibrators (live models were checkpointed immediately after fitting);
- an exact runtime requirements file;
- experiment configuration and environment metadata;
- `git log --oneline --all` when the repository is available;
- a machine-readable checklist.

Saving row-level predictions is essential: it allows every plot and metric to be audited without retraining.

In [ ]:
import shutil

# Row-level Task C predictions.
pred_c = pre_split["test"][["match_id", "kickoff", "home_team_name", "away_team_name",
                            "label_outcome", "label_margin"]].copy()
for name in cls_models:
    safe = name.lower().replace(" ", "_").replace("-", "_")
    pred_c[[f"{safe}_p_away", f"{safe}_p_draw", f"{safe}_p_home"]] = cls_test_cal[name]
pred_c[["market_p_away", "market_p_draw", "market_p_home"]] = market_test
pred_c.to_parquet(OUTPUT_DIR / "task_c_test_predictions.parquet", index=False)

# Row-level Task R predictions.
pred_r = pre_split["test"][["match_id", "kickoff", "home_team_name", "away_team_name",
                            "label_margin"]].copy()
for name, values in reg_test_pred.items():
    pred_r[name] = values
pred_r["NGBoost_sigma_raw"] = ngb_sigma_raw
pred_r["NGBoost_sigma_calibrated"] = ngb_sigma
pred_r.to_parquet(OUTPUT_DIR / "task_r_test_predictions.parquet", index=False)

# Row-level Task L predictions.
pred_l = live_split["test"][["match_id", "snapshot_id", "snapshot_rank",
                             "label_outcome", "label_margin"]].copy()
for name in live_cls_models:
    safe = name.lower().replace(" ", "_")
    pred_l[[f"{safe}_p_away", f"{safe}_p_draw", f"{safe}_p_home"]] = live_cls_test_cal[name]
for name, values in live_reg_test_pred.items():
    pred_l[f"{name.lower().replace(' ', '_')}_margin"] = values
pred_l.to_parquet(OUTPUT_DIR / "task_l_test_predictions.parquet", index=False)

if SAVE_MODELS:
    for name, model in cls_models.items():
        safe = name.lower().replace(" ", "_").replace("-", "_")
        joblib.dump({"model": model, "calibrator": cls_calibrators[name]},
                    MODEL_DIR / f"task_c_{safe}.joblib")
    for name, model in reg_models.items():
        safe = name.lower().replace(" ", "_").replace("-", "_")
        joblib.dump(model, MODEL_DIR / f"task_r_{safe}.joblib")
    if RUN_MULTIVARIATE:
        joblib.dump({"preprocessor": mv_prep, "model": mv_model},
                    MODEL_DIR / "p2_multivariate_ngboost.joblib")

packages = ["numpy", "pandas", "scipy", "pyarrow", "scikit-learn", "xgboost",
            "lightgbm", "ngboost", "imbalanced-learn", "shap", "seaborn", "psutil",
            "joblib", "Jinja2"]
versions = {}
for package in packages:
    try:
        versions[package] = ilm.version(package)
    except ilm.PackageNotFoundError:
        versions[package] = "not-installed"
(OUTPUT_DIR / "requirements_runtime.txt").write_text(
    "\n".join(f"{name}=={version}" for name, version in versions.items()) + "\n", encoding="utf-8"
)

run_config = {
    "seed": SEED, "run_mode": RUN_MODE, "candidates_per_family": N_CANDIDATES,
    "run_shap": RUN_SHAP, "run_multivariate": RUN_MULTIVARIATE,
    "ngboost_regression_selection": "validation_NLL",
    "ngboost_uncertainty_scale_source": "validation residuals only",
    "ngboost_sigma_scale": ngb_sigma_scale,
    "project_root": str(PROJECT_ROOT), "result_root": str(RESULT_ROOT),
    "split_matches": EXPECTED_SPLITS,
    "representation": "MD1_PLUS_P1_HOMEAWAY",
    "p1_selected_n": p1_experiment_summary["selected_n"],
    "p1_feature_count": len(p1_features),
    "md1_pre_features": md1_pre_features, "p1_features": p1_features,
    "pre_features": pre_features, "live_features": live_features,
    "class_order": CLASS_CODES,
}
(OUTPUT_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")
(OUTPUT_DIR / "environment.json").write_text(json.dumps({
    "python": sys.version, "platform": platform.platform(), "packages": versions,
}, indent=2), encoding="utf-8")

try:
    git_log = subprocess.run(
        ["git", "-c", f"safe.directory={PROJECT_ROOT.as_posix()}",
         "log", "--oneline", "--all"], cwd=PROJECT_ROOT,
        check=True, capture_output=True, text=True,
    ).stdout
except Exception as exc:
    git_log = f"Git history unavailable: {type(exc).__name__}: {exc}\n"
(OUTPUT_DIR / "git_log_oneline_all.txt").write_text(git_log, encoding="utf-8")

ngboost_regression_selection_rows = compute_table.loc[
    compute_table["Model"].eq("NGBoost") & compute_table["Task"].isin(["R", "L-regression"])
]
checklist = {
    "fixed_seed": True,
    "chronological_match_level_split": True,
    "p1_upstream_readiness_passed": bool(audit_boolean(p1_readiness["passed"]).all()),
    "p1_upstream_leakage_passed": bool(audit_boolean(p1_leakage["passed"]).all()),
    "p1_exp01_readiness_passed": bool(audit_boolean(exp01_readiness["passed"]).all()),
    "p1_validation_frozen_representation_loaded": p1_validation_decision["preferred_project_representation_by_validation_rps"] == "MD1_PLUS_P1_HOMEAWAY",
    "p1_18_feature_contract": len(p1_features) == 18,
    "p1_inherited_into_live_snapshots": set(p1_features).issubset(X_live["train"].columns),
    "training_only_transforms": True,
    "market_baseline_same_test_matches": True,
    "all_required_task_c_families": set(["Dummy", "SVC-RBF", "Random Forest", "GBM", "XGBoost", "LightGBM", "NGBoost"]).issubset(cls_models),
    "all_required_task_r_families": set(["Dummy", "SVR-RBF", "Kernel Ridge exact", "Kernel Ridge Nystroem", "Random Forest", "GBM", "XGBoost", "LightGBM", "NGBoost"]).issubset(reg_models),
    "all_required_task_l_families": set(["Dummy", "Random Forest", "GBM", "XGBoost", "LightGBM", "NGBoost"]).issubset(live_cls_models),
    "ngboost_regression_selected_by_validation_nll": bool(
        len(ngboost_regression_selection_rows) == 2
        and ngboost_regression_selection_rows["SelectionCriterion"].eq("validation_NLL").all()
    ),
    "ngboost_validation_only_scale_calibration": bool(np.isfinite(ngb_sigma_scale) and ngb_sigma_scale > 0),
    "calibration_before_after": True,
    "phase_calibration": True,
    "minute_metric_curves": True,
    "imbalance_arms": set(["Vanilla", "SMOTE", "BorderlineSMOTE", "ADASYN", "Class weights"]).issubset(imbalance_models),
    "paper_multivariate_experiment": bool(RUN_MULTIVARIATE),
    "paper_multivariate_marginal_and_joint_coverage": bool(
        RUN_MULTIVARIATE and len(mv_coverage) == 3
    ),
    "shap_requested_and_completed": bool(RUN_SHAP and len(shap_contexts) >= 12),
    "ten_worst_each_task": True,
    "compute_and_memory_table": True,
    "kernel_scaling": True,
}
(OUTPUT_DIR / "completion_checklist.json").write_text(json.dumps(checklist, indent=2), encoding="utf-8")
display(pd.Series(checklist, name="complete").to_frame())

archive_base = str(OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_evidence")
archive_path = shutil.make_archive(archive_base, "zip", root_dir=OUTPUT_DIR)
print("Evidence folder:", OUTPUT_DIR)
print("Evidence ZIP:", archive_path)

## 28. Limitations and honest interpretation checklist

Before writing the final conclusion, address these points explicitly:

1. **Sample size and shift:** the target season has 380 matches, but the frozen P1 eligibility rule leaves 374 modeling examples; the chronological test contains 46 matches and unusually few away wins. Rankings may not reproduce across seasons.
2. **Market ceiling:** bookmaker odds encode injuries, lineups, transfers, public information, and market behavior absent from event-derived rolling features. Failing to beat the market is not an implementation failure.
3. **Gaussian mismatch:** margins are bounded discrete values and goals are non-negative counts. Normal and multivariate Normal NGBoost are useful approximations, but a discrete bivariate count distribution would be more faithful.
4. **Snapshot correlation:** match-level splitting prevents leakage, yet standard per-snapshot averages still weight every match 21 times. Report match-clustered uncertainty or bootstrap by match in an extended study.
5. **Calibration uncertainty:** 46 test matches and phase subsets are too small for precise reliability estimates. The Task R scalar correction addresses global dispersion only; it cannot repair incorrect match-specific relative scales. Interpret coverage and reliability curves alongside counts.
6. **SMOTE physics:** resampling is studied only pre-match. Cross-match synthetic live states are deliberately prohibited.
7. **SHAP scope:** attributions explain model behavior, not causal football mechanisms; raw tree outputs are explained before Platt calibration.
8. **P1 boundary:** the 18 Berrar fields are genuinely pre-match. Repeating them across Task L snapshots is a declared project extension, not a claim that the P1 paper proposed an event-stream model. Full P1 fidelity and leakage evidence remains in the upstream deliverable.

An honest report should explain negative results, not delete them. The exported tables contain every metric and every attempted candidate so the final narrative can be audited.

## References

- Berrar, D., Lopes, P., & Dubitzky, W. (2024). *A data- and knowledge-driven framework for developing machine learning models to predict soccer match outcomes*. Machine Learning, 113, 8165–8204. https://doi.org/10.1007/s10994-024-06625-9
- O'Malley, M., Sykulski, A. M., Lumpkin, R., & Schuler, A. (2023). *Probabilistic Prediction of Oceanographic Velocities with Multivariate Gaussian Natural Gradient Boosting*. Environmental Data Science, 2, e10. https://doi.org/10.1017/eds.2023.4
- Duan, T., Avati, A., Ding, D. Y., Thai, K. K., Basu, S., Ng, A. Y., & Schuler, A. (2020). *NGBoost: Natural Gradient Boosting for Probabilistic Prediction*. Proceedings of ICML 2020, PMLR 119, 2690–2700.
- O'Malley et al. replication repository (release v0.2): https://github.com/MikeOMa/MV_Prediction
- Maintained NGBoost package and source: https://github.com/stanfordmlgroup/ngboost
- StatsBomb Open Data: https://github.com/statsbomb/open-data
- Football-Data.co.uk historical results and odds: https://www.football-data.co.uk/
- Lundberg, S. M., & Lee, S.-I. (2017). *A Unified Approach to Interpreting Model Predictions*. NeurIPS 2017.

**StatsBomb attribution:** data are used under the StatsBomb Open Data terms for non-commercial research and education. When publishing figures or results, include the StatsBomb logo/attribution required by their repository license guidance.